In [ ]:

# ============================================================
# SINUS 3D VIEWER PRO — CELL 1
# ENVIRONMENT SETUP + GOOGLE DRIVE
# ============================================================

import os
import sys
import subprocess
import platform

print("=" * 65)
print("SINUS 3D VIEWER PRO — INITIAL SETUP")
print("=" * 65)

packages = [
    "SimpleITK",
    "pydicom",
    "numpy",
    "scipy",
    "fastapi",
    "uvicorn",
    "orjson",
    "zstandard",
    "psutil"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", *packages
])

# from google.colab import drive
# drive.mount("/content/drive")

import torch
import psutil
import SimpleITK as sitk
import pydicom
import fastapi
import uvicorn

print("\nSYSTEM INFORMATION")
print("-" * 65)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print(
    "System RAM:",
    round(psutil.virtual_memory().total / 1024**3, 2),
    "GB"
)

print("SimpleITK:", sitk.Version_VersionString())

print("\nGOOGLE DRIVE")
print("-" * 65)
print("Mounted:", os.path.isdir("/content/drive/MyDrive"))

print("\nORIGINAL DATASET SAFETY")
print("Read-only inspection policy: ENABLED")
print("No patient files modified.")

print("\n" + "=" * 65)
print("CELL 1 COMPLETE")
print("=" * 65)


SINUS 3D VIEWER PRO — INITIAL SETUP

SYSTEM INFORMATION
-----------------------------------------------------------------
Python: 3.13.16
PyTorch: 2.11.0+cpu
CUDA available: False
System RAM: 50.99 GB
SimpleITK: 2.5.6

GOOGLE DRIVE
-----------------------------------------------------------------
Mounted: True

ORIGINAL DATASET SAFETY
Read-only inspection policy: ENABLED
No patient files modified.

CELL 1 COMPLETE


In [ ]:

# ============================================================
# SINUS 3D VIEWER PRO — CELL 2
# SMART DICOM SCANNER + SERIES SELECTOR
# STRICTLY READ-ONLY
# ============================================================

import os
import time
from pathlib import Path

import SimpleITK as sitk
import numpy as np

print("=" * 70)
print("SINUS 3D VIEWER PRO — PATIENT DICOM SCANNER")
print("=" * 70)

print("\nPaste patient folder FULL PATH.")
print("Example: /content/drive/MyDrive/Sinus_Project/Sinus/PATIENT_FOLDER")

raw_path = input("\nPATIENT FOLDER PATH: ").strip().strip('"').strip("'")
PATIENT_PATH = Path(raw_path).expanduser()

if not PATIENT_PATH.is_dir():
    raise FileNotFoundError(
        f"Patient folder not found: {PATIENT_PATH}"
    )

print("\n[1/5] Scanning folders...")
start_time = time.time()

DICOM_SERIES = []
scanned_folders = 0
scan_errors = []

for root, dirs, files in os.walk(PATIENT_PATH):
    dirs.sort()

    if not files:
        continue

    scanned_folders += 1

    try:
        ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(root)

        if not ids:
            continue

        for series_id in ids:
            paths = list(
                sitk.ImageSeriesReader.GetGDCMSeriesFileNames(
                    root, series_id
                )
            )

            if not paths:
                continue

            DICOM_SERIES.append({
                "folder": root,
                "series_id": series_id,
                "files": paths,
                "count": len(paths)
            })

    except Exception as exc:
        scan_errors.append((root, str(exc)[:150]))

if not DICOM_SERIES:
    raise RuntimeError(
        "No readable DICOM image series found. "
        "Check the folder path and DICOM files."
    )

DICOM_SERIES.sort(
    key=lambda item: item["count"],
    reverse=True
)

print("\n[2/5] Available DICOM series")
print("-" * 70)

for index, item in enumerate(DICOM_SERIES):
    folder_name = Path(item["folder"]).name

    print(
        f"[{index}] "
        f"{item['count']} files | "
        f"Folder: {folder_name}"
    )

print("-" * 70)
print("Total series:", len(DICOM_SERIES))
print("Folders scanned:", scanned_folders)
print("Folders with scan errors:", len(scan_errors))

print("\n[3/5] Series selection")
print("Default = 0 (series with most files)")

choice = input("Select series number [0]: ").strip()

try:
    selected_index = int(choice) if choice else 0
except ValueError:
    raise ValueError("Series selection must be a number.")

if not 0 <= selected_index < len(DICOM_SERIES):
    raise ValueError("Invalid series number.")

SELECTED_DICOM_SERIES = DICOM_SERIES[selected_index]

print("\n[4/5] Loading selected series into RAM...")

reader = sitk.ImageSeriesReader()
reader.SetFileNames(SELECTED_DICOM_SERIES["files"])

reader.MetaDataDictionaryArrayUpdateOn()
reader.LoadPrivateTagsOff()

CT_IMAGE = reader.Execute()

# SimpleITK image = X,Y,Z
# NumPy image = Z,Y,X
CT_ARRAY = sitk.GetArrayFromImage(CT_IMAGE)

CT_SIZE = CT_IMAGE.GetSize()
CT_SPACING = CT_IMAGE.GetSpacing()
CT_ORIGIN = CT_IMAGE.GetOrigin()
CT_DIRECTION = CT_IMAGE.GetDirection()

print("\n[5/5] Validating volume...")

if CT_ARRAY.ndim != 3:
    raise RuntimeError(
        f"Expected 3D volume, received {CT_ARRAY.ndim}D."
    )

if any(value <= 0 for value in CT_SPACING):
    raise RuntimeError("Invalid voxel spacing.")

finite_mask = np.isfinite(CT_ARRAY)

if not finite_mask.all():
    raise RuntimeError("Non-finite voxel values detected.")

# Check for orthonormal physical direction matrix.
direction_matrix = np.array(CT_DIRECTION).reshape(3, 3)
orthogonal_error = np.max(
    np.abs(direction_matrix.T @ direction_matrix - np.eye(3))
)

if orthogonal_error > 0.01:
    print("WARNING: Unusual image direction matrix.")

# Store global viewer-ready context.
SINUS_VIEWER_CONTEXT = {
    "patient_folder": str(PATIENT_PATH),
    "series_index": selected_index,
    "file_count": SELECTED_DICOM_SERIES["count"],
    "shape_zyx": CT_ARRAY.shape,
    "spacing_xyz": CT_SPACING,
    "origin_xyz": CT_ORIGIN,
    "direction": CT_DIRECTION,
    "volume_loaded": True
}

memory_gib = CT_ARRAY.nbytes / (1024 ** 3)
scan_seconds = time.time() - start_time

print("\n" + "=" * 70)
print("DICOM VOLUME REPORT")
print("=" * 70)

print("Selected series       :", selected_index)
print("DICOM files           :", SELECTED_DICOM_SERIES["count"])
print("Volume dimensions XYZ :", CT_SIZE)
print("NumPy dimensions ZYX  :", CT_ARRAY.shape)
print("Voxel spacing XYZ (mm):", tuple(round(x, 4) for x in CT_SPACING))
print("Data type             :", CT_ARRAY.dtype)
print("Minimum intensity     :", float(CT_ARRAY.min()))
print("Maximum intensity     :", float(CT_ARRAY.max()))
print("Volume RAM usage      :", round(memory_gib, 3), "GiB")
print("Scan + load time      :", round(scan_seconds, 2), "seconds")

if CT_ARRAY.shape[0] < 2:
    print("\nWARNING: Only one slice detected.")
    print("Full 3D reconstruction requires multiple slices.")

if len(DICOM_SERIES) > 1:
    print("\nNOTE: Multiple series found.")
    print("Verify that you selected the intended CT acquisition.")

print("\nORIGINAL DATASET SAFETY")
print("Source files modified : NO")
print("Files renamed         : NO")
print("Files moved           : NO")
print("Files deleted         : NO")

print("\n" + "=" * 70)
print("CELL 2 COMPLETE — VOLUME LOADED INTO RAM")
print("=" * 70)


SINUS 3D VIEWER PRO — PATIENT DICOM SCANNER

Paste patient folder FULL PATH.
Example: /content/drive/MyDrive/Sinus_Project/Sinus/PATIENT_FOLDER

PATIENT FOLDER PATH: /content/drive/MyDrive/Sinus_Project/Sinus/71-80/BHUPENDRA_M_274094

[1/5] Scanning folders...

[2/5] Available DICOM series
----------------------------------------------------------------------
[0] 768 files | Folder: BHUPENDRA_M_274094
----------------------------------------------------------------------
Total series: 1
Folders scanned: 1
Folders with scan errors: 0

[3/5] Series selection
Default = 0 (series with most files)
Select series number [0]: 0

[4/5] Loading selected series into RAM...

[5/5] Validating volume...

DICOM VOLUME REPORT
Selected series       : 0
DICOM files           : 768
Volume dimensions XYZ : (640, 640, 768)
NumPy dimensions ZYX  : (768, 640, 640)
Voxel spacing XYZ (mm): (0.08, 0.08, 0.08)
Data type             : int16
Minimum intensity     : -1024.0
Maximum intensity     : 2903.0
Volume RAM

In [ ]:
# ============================================================
# SINUS 3D VIEWER PRO — CELL 3
# RUNTIME PREPARATION + OPTIMIZED VIEWER DATA
# STRICTLY READ-ONLY
# ============================================================

import os
import io
import json
import math
import shutil
from pathlib import Path

import numpy as np
import SimpleITK as sitk
from scipy.ndimage import zoom

print("=" * 72)
print("SINUS 3D VIEWER PRO — RUNTIME PREPARATION")
print("=" * 72)

# -----------------------------
# Required globals check
# -----------------------------
required_globals = ["CT_ARRAY", "CT_IMAGE", "SINUS_VIEWER_CONTEXT"]
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required objects from Cell 2: {missing}"
    )

# -----------------------------
# Runtime folder
# -----------------------------
RUNTIME_DIR = Path("/content/sinus_viewer_runtime")
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = RUNTIME_DIR / "data"
STATIC_DIR = RUNTIME_DIR / "static"
DATA_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

print("\n[1/7] Copying volume into viewer runtime...")
volume_i16 = np.asarray(CT_ARRAY, dtype=np.int16)   # Z,Y,X
zdim, ydim, xdim = volume_i16.shape

spacing_xyz = tuple(float(x) for x in CT_IMAGE.GetSpacing())
origin_xyz = tuple(float(x) for x in CT_IMAGE.GetOrigin())
direction = tuple(float(x) for x in CT_IMAGE.GetDirection())

# -----------------------------
# HU window helper
# -----------------------------
def window_to_uint8(arr, level, width):
    low = level - width / 2.0
    high = level + width / 2.0
    clipped = np.clip(arr, low, high)
    scaled = ((clipped - low) / (high - low) * 255.0).astype(np.uint8)
    return scaled

# -----------------------------
# Thumbnail previews
# -----------------------------
print("[2/7] Generating quick preview thumbnails...")

mid_z = zdim // 2
mid_y = ydim // 2
mid_x = xdim // 2

preview_axial = window_to_uint8(volume_i16[mid_z], 400, 2000)
preview_coronal = window_to_uint8(volume_i16[:, mid_y, :], 400, 2000)
preview_sagittal = window_to_uint8(volume_i16[:, :, mid_x], 400, 2000)

np.save(DATA_DIR / "preview_axial.npy", preview_axial)
np.save(DATA_DIR / "preview_coronal.npy", preview_coronal)
np.save(DATA_DIR / "preview_sagittal.npy", preview_sagittal)

# -----------------------------
# 3D volume downsampling
# -----------------------------
print("[3/7] Preparing optimized 3D volume...")

# Target maximum side for browser-friendly 3D
target_max_side = 192

scale_z = min(1.0, target_max_side / zdim)
scale_y = min(1.0, target_max_side / ydim)
scale_x = min(1.0, target_max_side / xdim)

scale = min(scale_z, scale_y, scale_x)

if scale < 1.0:
    vol_small = zoom(
        volume_i16.astype(np.float32),
        zoom=(scale, scale, scale),
        order=1
    ).astype(np.float32)
else:
    vol_small = volume_i16.astype(np.float32)

# Bone-friendly normalization for volume rendering
# We keep air/bone visibility balanced
vol_small_clip = np.clip(vol_small, -1024, 2000)
vol_small_u8 = ((vol_small_clip + 1024.0) / (2000.0 + 1024.0) * 255.0)
vol_small_u8 = np.clip(vol_small_u8, 0, 255).astype(np.uint8)

np.save(DATA_DIR / "volume3d_u8.npy", vol_small_u8)

scaled_spacing_xyz = (
    spacing_xyz[0] * (xdim / vol_small_u8.shape[2]),
    spacing_xyz[1] * (ydim / vol_small_u8.shape[1]),
    spacing_xyz[2] * (zdim / vol_small_u8.shape[0]),
)

# -----------------------------
# Save main volume for slice service
# -----------------------------
print("[4/7] Saving full-resolution volume for slice streaming...")
np.save(DATA_DIR / "volume_full_i16.npy", volume_i16)

# -----------------------------
# Metadata
# -----------------------------
print("[5/7] Writing viewer metadata...")

viewer_metadata = {
    "patient_folder": SINUS_VIEWER_CONTEXT["patient_folder"],
    "series_index": int(SINUS_VIEWER_CONTEXT["series_index"]),
    "file_count": int(SINUS_VIEWER_CONTEXT["file_count"]),
    "shape_zyx": [int(zdim), int(ydim), int(xdim)],
    "spacing_xyz_mm": [float(x) for x in spacing_xyz],
    "origin_xyz": [float(x) for x in origin_xyz],
    "direction": [float(x) for x in direction],
    "dtype": str(volume_i16.dtype),
    "hu_min": int(volume_i16.min()),
    "hu_max": int(volume_i16.max()),
    "default_window": {
        "bone": {"level": 400, "width": 2000},
        "sinus": {"level": 300, "width": 1500},
        "soft_tissue": {"level": 40, "width": 400},
        "air": {"level": -600, "width": 1500},
    },
    "mid_slices": {
        "axial": int(mid_z),
        "coronal": int(mid_y),
        "sagittal": int(mid_x),
    },
    "volume3d_shape_zyx": [int(v) for v in vol_small_u8.shape],
    "volume3d_spacing_xyz_mm": [float(x) for x in scaled_spacing_xyz],
    "runtime_dir": str(RUNTIME_DIR),
    "notes": [
        "Original DICOM files remain unchanged.",
        "2D views stream full-resolution slices on demand.",
        "3D volume uses optimized downsampled representation.",
    ],
}

with open(DATA_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(viewer_metadata, f, indent=2)

# -----------------------------
# Global context for next cells
# -----------------------------
SINUS_VIEWER_RUNTIME = {
    "runtime_dir": str(RUNTIME_DIR),
    "data_dir": str(DATA_DIR),
    "static_dir": str(STATIC_DIR),
    "metadata_path": str(DATA_DIR / "metadata.json"),
    "full_volume_path": str(DATA_DIR / "volume_full_i16.npy"),
    "volume3d_path": str(DATA_DIR / "volume3d_u8.npy"),
    "shape_zyx": volume_i16.shape,
    "volume3d_shape_zyx": vol_small_u8.shape,
}

# -----------------------------
# Report
# -----------------------------
full_mem_gib = volume_i16.nbytes / (1024**3)
small_mem_gib = vol_small_u8.nbytes / (1024**3)

print("[6/7] Runtime summary")
print("-" * 72)
print("Full volume shape (ZYX)     :", volume_i16.shape)
print("Full volume dtype           :", volume_i16.dtype)
print("Full volume RAM size        :", round(full_mem_gib, 3), "GiB")
print("3D optimized shape (ZYX)    :", vol_small_u8.shape)
print("3D optimized dtype          :", vol_small_u8.dtype)
print("3D optimized RAM size       :", round(small_mem_gib, 3), "GiB")
print("Runtime folder              :", RUNTIME_DIR)

print("\n[7/7] Dataset safety check")
print("-" * 72)
print("Original DICOM files modified : NO")
print("Original DICOM files moved    : NO")
print("Original DICOM files renamed  : NO")
print("Original DICOM files deleted  : NO")
print("Temporary runtime generated   : YES")

print("\n" + "=" * 72)
print("CELL 3 COMPLETE — VIEWER RUNTIME READY")
print("=" * 72)

SINUS 3D VIEWER PRO — RUNTIME PREPARATION

[1/7] Copying volume into viewer runtime...
[2/7] Generating quick preview thumbnails...
[3/7] Preparing optimized 3D volume...
[4/7] Saving full-resolution volume for slice streaming...
[5/7] Writing viewer metadata...
[6/7] Runtime summary
------------------------------------------------------------------------
Full volume shape (ZYX)     : (768, 640, 640)
Full volume dtype           : int16
Full volume RAM size        : 0.586 GiB
3D optimized shape (ZYX)    : (192, 160, 160)
3D optimized dtype          : uint8
3D optimized RAM size       : 0.005 GiB
Runtime folder              : /content/sinus_viewer_runtime

[7/7] Dataset safety check
------------------------------------------------------------------------
Original DICOM files modified : NO
Original DICOM files moved    : NO
Original DICOM files renamed  : NO
Original DICOM files deleted  : NO
Temporary runtime generated   : YES

CELL 3 COMPLETE — VIEWER RUNTIME READY


In [ ]:

# ================================================================
# SINUS 3D VIEWER PRO — CELL 4
# FASTAPI + 3 MPR PLANES + INTERACTIVE VTK.JS VOLUME
# TEMPORARY RUNTIME — ORIGINAL DICOM FILES READ-ONLY
# ================================================================

import io, json, os, secrets, threading, time, socket, hmac
from pathlib import Path

import numpy as np
from PIL import Image
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import (
    HTMLResponse, JSONResponse, Response, RedirectResponse
)
from starlette.middleware.base import BaseHTTPMiddleware
import uvicorn

if "SINUS_VIEWER_RUNTIME" not in globals():
    raise RuntimeError("Pehle Cell 3 run karo.")

DATA_PATH = Path(SINUS_VIEWER_RUNTIME["data_dir"])
with open(DATA_PATH / "metadata.json") as f:
    META = json.load(f)

# Memory-map full resolution CT for efficient slice access.
FULL = np.load(
    DATA_PATH / "volume_full_i16.npy",
    mmap_mode="r"
)

SMALL_PATH = DATA_PATH / "volume3d_u8.npy"
Z, Y, X = FULL.shape

# Random per-session credentials.
VIEWER_PASSWORD = secrets.token_urlsafe(18)
VIEWER_SESSION = secrets.token_urlsafe(32)
VIEWER_PORT = 8765

APP = FastAPI(
    title="Sinus 3D Viewer Pro",
    docs_url=None,
    redoc_url=None,
    openapi_url=None
)

def is_authorized(request):
    cookie = request.cookies.get("sinus_session", "")
    return hmac.compare_digest(cookie, VIEWER_SESSION)

class AuthMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        allowed = {"/login", "/health"}
        if request.url.path in allowed:
            return await call_next(request)
        if not is_authorized(request):
            if request.url.path.startswith("/api/"):
                return JSONResponse(
                    {"error": "Authentication required"},
                    status_code=401
                )
            return RedirectResponse("/login")
        return await call_next(request)

APP.add_middleware(AuthMiddleware)

LOGIN_HTML = """
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sinus Viewer Login</title>
<style>
*{box-sizing:border-box}
body{margin:0;min-height:100vh;display:grid;place-items:center;
background:#090f19;color:#e8f1ff;font-family:Inter,Segoe UI,Arial}
.panel{width:min(410px,92vw);padding:36px;background:#131e2e;
border:1px solid #293a50;border-radius:18px}
h1{font-size:24px;margin:0 0 12px}
p{color:#96a8bf;font-size:14px;line-height:1.6}
input,button{width:100%;padding:14px;border-radius:9px;font-size:15px}
input{background:#0b1421;border:1px solid #40516a;color:white}
button{margin-top:14px;border:0;background:#328cff;color:white;
font-weight:700;cursor:pointer}
</style>
</head>
<body><div class="panel">
<h1>Sinus 3D Viewer Pro</h1>
<p>Private temporary imaging session. Enter the password
printed by Cell 4 in Colab.</p>
<form method="post" action="/login">
<input name="password" type="password" autocomplete="off"
placeholder="Session password" required autofocus>
<button type="submit">Open Viewer</button>
</form>
</div></body>
</html>
"""

from fastapi import Form

@APP.get("/login", response_class=HTMLResponse)
def login_page():
    return LOGIN_HTML

@APP.post("/login")
def login_submit(password: str = Form(...)):
    if not hmac.compare_digest(password, VIEWER_PASSWORD):
        return HTMLResponse(
            LOGIN_HTML.replace(
                "Private temporary imaging session.",
                "Incorrect password. Try again."
            ),
            status_code=401
        )
    resp = RedirectResponse("/", status_code=303)
    resp.set_cookie(
        "sinus_session", VIEWER_SESSION,
        httponly=True, samesite="strict",
        max_age=3600
    )
    return resp

@APP.get("/health")
def health():
    return {"status": "running"}

@APP.get("/api/meta")
def metadata():
    # No identifying patient name or source folder exposed.
    return {
        "shape": [Z, Y, X],
        "spacing": META["spacing_xyz_mm"],
        "direction": META["direction"],
        "origin": META["origin_xyz"],
        "fileCount": META["file_count"],
        "volumeShape": META["volume3d_shape_zyx"],
        "volumeSpacing": META["volume3d_spacing_xyz_mm"]
    }

def make_window(image, level, width):
    low = level - width / 2
    normalized = (image.astype(np.float32) - low) / width
    return np.uint8(np.clip(normalized, 0, 1) * 255)

@APP.get("/api/slice/{plane}/{index}")
def fetch_slice(
    plane: str,
    index: int,
    level: float = 400,
    width: float = 2000
):
    if not (-1100 <= level <= 3200 and 1 <= width <= 6000):
        raise HTTPException(400, "Invalid window parameters")

    if plane == "axial":
        if not 0 <= index < Z:
            raise HTTPException(400, "Invalid slice")
        data = FULL[index, :, :]
    elif plane == "coronal":
        if not 0 <= index < Y:
            raise HTTPException(400, "Invalid slice")
        data = FULL[:, index, :]
    elif plane == "sagittal":
        if not 0 <= index < X:
            raise HTTPException(400, "Invalid slice")
        data = FULL[:, :, index]
    else:
        raise HTTPException(404, "Unknown plane")

    # Flip vertical voxel axis for display.
    bitmap = Image.fromarray(
        make_window(np.flipud(data), level, width),
        mode="L"
    )
    output = io.BytesIO()
    bitmap.save(output, format="PNG", compress_level=1)

    return Response(
        output.getvalue(),
        media_type="image/png",
        headers={"Cache-Control": "no-store"}
    )

@APP.get("/api/volume")
def fetch_volume():
    return Response(
        SMALL_PATH.read_bytes()[128:] if False else
        np.load(SMALL_PATH).tobytes(order="C"),
        media_type="application/octet-stream",
        headers={"Cache-Control": "no-store"}
    )

VIEWER_HTML = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sinus 3D Viewer Pro</title>
<script src="https://unpkg.com/vtk.js/dist/vtk.js"></script>
<style>
:root{--bg:#080e17;--panel:#101a29;--line:#26364b;
--muted:#8ba0b9;--text:#e8f1ff;--blue:#389bff}
*{box-sizing:border-box}
html,body{height:100%;margin:0;background:var(--bg);
font-family:Inter,Segoe UI,Arial,sans-serif;color:var(--text)}
body{display:flex;flex-direction:column;overflow:hidden}
header{height:62px;display:flex;align-items:center;justify-content:space-between;
padding:0 19px;background:#101927;border-bottom:1px solid var(--line);gap:12px}
.brand{font-size:17px;font-weight:750;letter-spacing:.2px}
.brand span{color:#42a8ff}
.sub{font-size:11px;color:var(--muted);margin-top:3px}
button,select,input[type=range]{cursor:pointer}
button,select{background:#1a2a3d;color:var(--text);
border:1px solid #30435b;border-radius:7px;padding:8px 10px}
button:hover{border-color:#4b9de8}
.toolbar{display:flex;align-items:center;gap:8px;flex-wrap:wrap}
main{flex:1;min-height:0;display:grid;grid-template-columns:1fr 1fr;
grid-template-rows:1fr 1fr;gap:8px;padding:8px}
.panel{min-width:0;min-height:0;position:relative;overflow:hidden;
border-radius:11px;background:#02060c;border:1px solid var(--line)}
.topbar{position:absolute;top:0;left:0;right:0;z-index:4;display:flex;
align-items:center;justify-content:space-between;padding:9px 12px;
background:linear-gradient(#091421ed,#09142177)}
.tag{font-size:11px;font-weight:750;letter-spacing:1.5px}
.count{font-size:11px;color:#a9bfda;font-variant-numeric:tabular-nums}
.viewport{position:absolute;inset:0;display:flex;align-items:center;
justify-content:center;touch-action:none;overflow:hidden}
.viewport canvas{display:block;max-width:none;image-rendering:auto}
.vtkview{position:absolute;inset:0;overflow:hidden}
.controls{position:absolute;bottom:9px;left:12px;right:12px;z-index:5;
display:flex;align-items:center;gap:10px;
background:#091522db;padding:8px 12px;border-radius:8px}
.controls input{flex:1;min-width:20px;accent-color:var(--blue)}
.status{font-size:11px;color:var(--muted)}
.foot{height:30px;display:flex;justify-content:space-between;
align-items:center;padding:0 16px;color:#92a5bb;font-size:11px;
border-top:1px solid var(--line)}
.full{position:fixed!important;inset:8px!important;z-index:99!important}
.panel:fullscreen{width:100vw;height:100vh;border-radius:0}
@media(max-width:780px){
header{height:auto;min-height:85px;align-items:flex-start;padding:10px}
main{grid-template-columns:1fr;grid-template-rows:repeat(4,minmax(300px,1fr));
overflow:auto}.toolbar{justify-content:flex-end}
}
</style>
</head>
<body>
<header>
<div><div class="brand"><span>◈</span> SINUS 3D VIEWER <span>PRO</span></div>
<div class="sub" id="scanInfo">Loading scan geometry...</div></div>
<div class="toolbar">
<select id="preset">
<option value="400,2000">Bone / Sinus</option>
<option value="40,400">Soft Tissue</option>
<option value="-600,1500">Air / Lung</option>
<option value="300,1500">Sinus Detail</option>
</select>
<button id="reset">Reset 2D</button>
<button id="reset3d">Reset 3D</button>
</div>
</header>
<main>
<section class="panel" id="p-axial">
<div class="topbar"><span class="tag" style="color:#f68b90">AXIAL</span>
<div><span class="count" id="n-axial"></span>
<button onclick="fullscreen('p-axial')">⛶</button></div></div>
<div class="viewport" id="v-axial"><canvas id="c-axial"></canvas></div>
<div class="controls"><input type="range" id="s-axial"><span class="status">Wheel: slices</span></div>
</section>

<section class="panel" id="p-coronal">
<div class="topbar"><span class="tag" style="color:#69dba0">CORONAL</span>
<div><span class="count" id="n-coronal"></span>
<button onclick="fullscreen('p-coronal')">⛶</button></div></div>
<div class="viewport" id="v-coronal"><canvas id="c-coronal"></canvas></div>
<div class="controls"><input type="range" id="s-coronal"><span class="status">Wheel: slices</span></div>
</section>

<section class="panel" id="p-sagittal">
<div class="topbar"><span class="tag" style="color:#efd16d">SAGITTAL</span>
<div><span class="count" id="n-sagittal"></span>
<button onclick="fullscreen('p-sagittal')">⛶</button></div></div>
<div class="viewport" id="v-sagittal"><canvas id="c-sagittal"></canvas></div>
<div class="controls"><input type="range" id="s-sagittal"><span class="status">Wheel: slices</span></div>
</section>

<section class="panel" id="p-3d">
<div class="topbar"><span class="tag" style="color:#77b5ff">3D VOLUME</span>
<div><span class="count" id="n-3d">Initializing...</span>
<button onclick="fullscreen('p-3d')">⛶</button></div></div>
<div id="view3d" class="vtkview"></div>
<div class="controls">
<span class="status">Bone opacity</span>
<input id="opacity" type="range" min="1" max="100" value="48">
<span class="status">Drag: rotate · Wheel: zoom</span>
</div>
</section>
</main>
<div class="foot"><span>Browser GPU · High-resolution 2D · Interactive 3D</span>
<span id="serverStatus">Connecting...</span></div>

<script>
const planes=["axial","coronal","sagittal"];
const images={}, state={}, controllers={};
let metadata, level=400,width=2000, vtkRenderer=null;
let vtkWindow=null, vtkOpacity=null;

function fullscreen(id){
 const el=document.getElementById(id);
 if(document.fullscreenElement) document.exitFullscreen();
 else el.requestFullscreen();
}
function clamp(x,a,b){return Math.max(a,Math.min(b,x));}
function status(s){document.getElementById("serverStatus").textContent=s;}

function draw(plane){
 const item=state[plane];
 if(!item || !item.img)return;
 const container=document.getElementById("v-"+plane);
 const canvas=document.getElementById("c-"+plane);
 const ctx=canvas.getContext("2d");
 const w=container.clientWidth,h=container.clientHeight;
 if(!w||!h)return;
 const dpr=Math.min(window.devicePixelRatio||1,2);
 canvas.width=Math.round(w*dpr);
 canvas.height=Math.round(h*dpr);
 canvas.style.width=w+"px";canvas.style.height=h+"px";
 ctx.setTransform(dpr,0,0,dpr,0,0);
 ctx.fillStyle="#02060c";ctx.fillRect(0,0,w,h);
 const iw=item.img.naturalWidth,ih=item.img.naturalHeight;
 // Physical aspect ratio, accounting for anisotropic voxels.
 const sx=metadata.spacing[plane==="sagittal"?1:0];
 const sy=metadata.spacing[plane==="axial"?1:2];
 const ar=(iw*sx)/(ih*sy);
 const fit=Math.min(w/ar,h)*.92;
 const dh=fit*item.zoom, dw=dh*ar;
 ctx.imageSmoothingEnabled=true;
 ctx.imageSmoothingQuality="high";
 ctx.drawImage(item.img,
   (w-dw)/2+item.dx,(h-dh)/2+item.dy,dw,dh);
}

async function loadSlice(plane){
 const item=state[plane];
 const seq=++item.seq;
 const url=`/api/slice/${plane}/${item.index}?level=${level}&width=${width}`;
 const img=new Image();
 img.onload=()=>{
  if(seq!==item.seq)return;
  item.img=img;draw(plane);
  document.getElementById("n-"+plane).textContent=
    `${item.index+1} / ${item.total}`;
 };
 img.onerror=()=>status("Slice loading error");
 img.src=url;
}

function initializePlanes(){
 const limits={axial:metadata.shape[0],
 coronal:metadata.shape[1],sagittal:metadata.shape[2]};
 planes.forEach(plane=>{
  const total=limits[plane];
  state[plane]={index:Math.floor(total/2),total,seq:0,
                  zoom:1,dx:0,dy:0,img:null};
  const slider=document.getElementById("s-"+plane);
  slider.min=0;slider.max=total-1;
  slider.value=state[plane].index;
  slider.oninput=()=>{
   state[plane].index=Number(slider.value);
   loadSlice(plane);
  };
  const surface=document.getElementById("v-"+plane);
  surface.addEventListener("wheel",e=>{
   e.preventDefault();
   const s=state[plane];
   if(e.ctrlKey){
    s.zoom=clamp(s.zoom*(e.deltaY<0?1.12:.89),.3,14);
    draw(plane);return;
   }
   s.index=clamp(s.index+(e.deltaY>0?1:-1),0,total-1);
   slider.value=s.index;
   loadSlice(plane);
  },{passive:false});
  let drag=null;
  surface.addEventListener("pointerdown",e=>{
   drag=[e.clientX,e.clientY];
   surface.setPointerCapture(e.pointerId);
  });
  surface.addEventListener("pointermove",e=>{
   if(!drag)return;
   const s=state[plane];
   s.dx+=e.clientX-drag[0];s.dy+=e.clientY-drag[1];
   drag=[e.clientX,e.clientY];draw(plane);
  });
  surface.addEventListener("pointerup",()=>drag=null);
  surface.addEventListener("pointercancel",()=>drag=null);
  surface.addEventListener("dblclick",()=>{
   const s=state[plane];s.zoom=1;s.dx=0;s.dy=0;draw(plane);
  });
  loadSlice(plane);
 });
}

async function initialize3D(){
 const label=document.getElementById("n-3d");
 try{
 if(!window.vtk)throw Error("vtk.js library not loaded");
 const shape=metadata.volumeShape;
 const spacing=metadata.volumeSpacing;
 const resp=await fetch("/api/volume");
 if(!resp.ok)throw Error("Volume download failed");
 const raw=new Uint8Array(await resp.arrayBuffer());
 if(raw.length!==shape[0]*shape[1]*shape[2])
   throw Error("3D volume dimensions mismatch");

 const V=vtk;
 const data=V.Common.DataModel.vtkImageData.newInstance();
 data.setDimensions(shape[2],shape[1],shape[0]);
 data.setSpacing(...spacing);
 // Spatial direction is retained; 2D medical orientation will
 // be separately validated before diagnostic interpretation.
 if(typeof data.setDirection==="function"){
   data.setDirection(metadata.direction);
 }
 const scalars=V.Common.Core.vtkDataArray.newInstance({
   name:"CTIntensity",values:raw,numberOfComponents:1
 });
 data.getPointData().setScalars(scalars);

 const mapper=V.Rendering.Core.vtkVolumeMapper.newInstance();
 mapper.setInputData(data);
 mapper.setSampleDistance(Math.min(...spacing)*1.3);

 const vol=V.Rendering.Core.vtkVolume.newInstance();
 vol.setMapper(mapper);

 const color=V.Rendering.Core.vtkColorTransferFunction.newInstance();
 // Values from Cell 3 are mapped to 0..255.
 color.addRGBPoint(0,0,0,0);
 color.addRGBPoint(70,.18,.24,.31);
 color.addRGBPoint(120,.65,.68,.72);
 color.addRGBPoint(190,.93,.89,.82);
 color.addRGBPoint(255,1,1,1);

 const opacity=V.Common.DataModel.vtkPiecewiseFunction.newInstance();
 vtkOpacity=opacity;

 function applyOpacity(){
   const strength=Number(document.getElementById("opacity").value)/100;
   opacity.removeAllPoints();
   opacity.addPoint(0,0);
   opacity.addPoint(105,0);
   opacity.addPoint(115,.002*strength);
   opacity.addPoint(140,.035*strength);
   opacity.addPoint(180,.19*strength);
   opacity.addPoint(255,.50*strength);
   if(vtkWindow)vtkWindow.render();
 }

 vol.getProperty().setRGBTransferFunction(0,color);
 vol.getProperty().setScalarOpacity(0,opacity);
 vol.getProperty().setInterpolationTypeToLinear();
 vol.getProperty().setShade(true);
 vol.getProperty().setAmbient(.25);
 vol.getProperty().setDiffuse(.75);
 vol.getProperty().setSpecular(.15);
 vol.getProperty().setScalarOpacityUnitDistance(0,Math.min(...spacing)*2);

 const fullscreenRenderer=
   V.Rendering.Misc.vtkFullScreenRenderWindow.newInstance({
    rootContainer:document.getElementById("view3d"),
    containerStyle:{height:"100%",width:"100%",position:"absolute"},
    background:[.008,.016,.029]
   });
 vtkRenderer=fullscreenRenderer.getRenderer();
 vtkWindow=fullscreenRenderer.getRenderWindow();
 vtkRenderer.addVolume(vol);
 vtkRenderer.resetCamera();
 applyOpacity();
 vtkWindow.render();
 document.getElementById("opacity").oninput=applyOpacity;
 label.textContent=shape.join(" × ")+" voxels";
 document.getElementById("reset3d").onclick=()=>{
   vtkRenderer.resetCamera();vtkWindow.render();
 };
 }catch(e){
  console.error(e);
  label.textContent="3D error: "+e.message;
  status("3D setup failed — see browser console");
 }
}

document.getElementById("preset").onchange=e=>{
 [level,width]=e.target.value.split(",").map(Number);
 planes.forEach(loadSlice);
};
document.getElementById("reset").onclick=()=>{
 planes.forEach(p=>{
  Object.assign(state[p],{zoom:1,dx:0,dy:0});
  draw(p);
 });
};

window.addEventListener("resize",()=>{
 planes.forEach(draw);
 if(vtkWindow)vtkWindow.render();
});
document.addEventListener("fullscreenchange",()=>{
 setTimeout(()=>{
  planes.forEach(draw);
  if(vtkWindow)vtkWindow.render();
 },100);
});

async function main(){
 try{
 const r=await fetch("/api/meta");
 if(!r.ok)throw Error("Metadata API failed");
 metadata=await r.json();
 document.getElementById("scanInfo").textContent=
   `${metadata.shape.join(" × ")} voxels · ${metadata.fileCount} DICOM slices`;
 initializePlanes();
 status("2D connected · Preparing 3D");
 await initialize3D();
 if(vtkWindow)status("Viewer online");
 }catch(e){
  status("Connection error: "+e.message);console.error(e);
 }
}
main();
</script>
</body>
</html>
"""

@APP.get("/", response_class=HTMLResponse)
def homepage():
    return HTMLResponse(
        VIEWER_HTML,
        headers={"Cache-Control": "no-store"}
    )

# Start or reuse backend.
if "SINUS_SERVER" in globals() and SINUS_SERVER.started:
    SINUS_SERVER.should_exit = True
    time.sleep(2)

config = uvicorn.Config(
    APP, host="127.0.0.1", port=VIEWER_PORT,
    log_level="warning", access_log=False
)
SINUS_SERVER = uvicorn.Server(config)
SINUS_SERVER_THREAD = threading.Thread(
    target=SINUS_SERVER.run, daemon=True
)
SINUS_SERVER_THREAD.start()

for _ in range(40):
    if SINUS_SERVER.started:
        break
    time.sleep(.25)

if not SINUS_SERVER.started:
    raise RuntimeError("Backend failed to start.")

import requests
response = requests.get(
    f"http://127.0.0.1:{VIEWER_PORT}/health", timeout=5
)
response.raise_for_status()

print("=" * 65)
print("SINUS 3D VIEWER PRO — CELL 4")
print("=" * 65)
print("Backend status : ONLINE")
print("Local address  : http://127.0.0.1:8765")
print("2D viewports   : Axial, Coronal, Sagittal")
print("3D viewport    : vtk.js WebGL Volume")
print("Original DICOM : UNCHANGED")
print()
print("PRIVATE SESSION PASSWORD:")
print(VIEWER_PASSWORD)
print()
print("Password kisi public chat ya screenshot mein share na karein.")
print("External HTTPS link will be configured separately.")
print("=" * 65)
print("CELL 4 COMPLETE")
print("=" * 65)


SINUS 3D VIEWER PRO — CELL 4
Backend status : ONLINE
Local address  : http://127.0.0.1:8765
2D viewports   : Axial, Coronal, Sagittal
3D viewport    : vtk.js WebGL Volume
Original DICOM : UNCHANGED

PRIVATE SESSION PASSWORD:
E0yjGpagTO0DcfP6do3NA7hA

Password kisi public chat ya screenshot mein share na karein.
External HTTPS link will be configured separately.
CELL 4 COMPLETE


In [ ]:

# ============================================================
# SINUS 3D VIEWER PRO — CELL 4B
# 3D ENGINE REPAIR + IMAGE CLARITY + DIAGNOSTICS
# EXISTING BACKEND / TUNNEL REUSED
# ============================================================

import requests
import secrets
import re
import numpy as np
from fastapi.responses import Response

print("=" * 68)
print("SINUS VIEWER PRO — 3D ENGINE REPAIR")
print("=" * 68)

if "APP" not in globals() or "VIEWER_HTML" not in globals():
    raise RuntimeError("Cell 4 backend context missing.")

if "SINUS_SERVER" not in globals() or not SINUS_SERVER.started:
    raise RuntimeError("Cell 4 server is not running.")

# ------------------------------------------------------------
# 1. Fetch known vtk.js browser bundle into Colab RAM
# ------------------------------------------------------------

print("\n[1/6] Loading vtk.js browser rendering engine...")

vtk_urls = [
    "https://cdn.jsdelivr.net/npm/vtk.js@35.7.1/vtk.js",
    "https://unpkg.com/vtk.js@35.7.1/vtk.js"
]

VTK_BUNDLE = None

for url in vtk_urls:
    try:
        r = requests.get(url, timeout=45)
        r.raise_for_status()

        if len(r.content) < 500_000:
            raise ValueError("Unexpectedly small JavaScript bundle")

        VTK_BUNDLE = r.content

        print("vtk.js downloaded successfully.")
        print("Bundle size:", round(len(VTK_BUNDLE) / 1048576, 2), "MiB")
        break

    except Exception as e:
        print("Source unavailable:", type(e).__name__)

if VTK_BUNDLE is None:
    raise RuntimeError(
        "Could not download vtk.js. Existing viewer unchanged."
    )

# Serve JS from existing authenticated backend.
# Add route only once; updated global bytes work on reruns.
if not globals().get("VTK_ROUTE_INSTALLED", False):
    @APP.get("/vendor/vtk.js")
    def serve_vtk_bundle():
        return Response(
            VTK_BUNDLE,
            media_type="application/javascript",
            headers={"Cache-Control": "no-store"}
        )

    VTK_ROUTE_INSTALLED = True

# ------------------------------------------------------------
# 2. Replace external JS library dependency
# ------------------------------------------------------------

print("[2/6] Replacing browser rendering engine...")

VIEWER_HTML = re.sub(
    r'<script src="https://unpkg\.com/vtk\.js/dist/vtk\.js"></script>',
    '<script src="/vendor/vtk.js"></script>',
    VIEWER_HTML
)

# Keep future execution idempotent.
VIEWER_HTML = VIEWER_HTML.replace(
    '<script src="https://cdn.jsdelivr.net/npm/vtk.js@35.7.1/vtk.js"></script>',
    '<script src="/vendor/vtk.js"></script>'
)

if '<script src="/vendor/vtk.js"></script>' not in VIEWER_HTML:
    raise RuntimeError("Could not locate vtk.js script tag.")

# ------------------------------------------------------------
# 3. Replace 3D initializer
# ------------------------------------------------------------

print("[3/6] Installing improved 3D volume renderer...")

NEW_3D_JS = r"""
let vtkFullScreen = null;
let vtkCameraRenderer = null;
let vtkRenderWindow = null;
let vtkVolumeProperty = null;

async function initialize3D() {
 const label = document.getElementById("n-3d");
 const panel = document.getElementById("view3d");

 function report(msg) {
   label.textContent = msg;
   console.log("[SINUS 3D]", msg);
 }

 try {
   report("Checking WebGL...");

   const testCanvas = document.createElement("canvas");
   const gl = testCanvas.getContext("webgl2", {
     failIfMajorPerformanceCaveat: false
   });

   if (!gl) {
     throw Error("WebGL2 unavailable. Enable browser hardware acceleration.");
   }

   if (!window.vtk) {
     throw Error("vtk.js failed to initialize.");
   }

   report("Downloading CT volume...");

   const response = await fetch("/api/volume", {
     cache: "no-store"
   });

   if (!response.ok) {
     throw Error("Volume API returned HTTP " + response.status);
   }

   const buffer = await response.arrayBuffer();
   const voxels = new Uint8Array(buffer);

   const dims = metadata.volumeShape;
   const spacing = metadata.volumeSpacing;
   const expected = dims[0] * dims[1] * dims[2];

   if (voxels.length !== expected) {
     throw Error(
       "Voxel mismatch: expected " + expected +
       ", got " + voxels.length
     );
   }

   report("Building 3D volume...");

   const V = window.vtk;

   // NumPy volume Z,Y,X -> vtkImageData X,Y,Z
   const image = V.Common.DataModel.vtkImageData.newInstance();

   image.setDimensions(dims[2], dims[1], dims[0]);
   image.setSpacing(spacing[0], spacing[1], spacing[2]);
   image.setOrigin(0, 0, 0);

   // Preserve direction cosines from original image metadata.
   if (typeof image.setDirection === "function") {
     image.setDirection(metadata.direction);
   }

   const scalars = V.Common.Core.vtkDataArray.newInstance({
     name: "CTVolume",
     values: voxels,
     numberOfComponents: 1
   });

   image.getPointData().setScalars(scalars);

   const mapper = V.Rendering.Core.vtkVolumeMapper.newInstance();
   mapper.setInputData(image);

   // Larger sample step improves interaction speed.
   mapper.setSampleDistance(
     Math.max(0.5, Math.min(...spacing) * 1.5)
   );

   const volume = V.Rendering.Core.vtkVolume.newInstance();
   volume.setMapper(mapper);

   // The uint8 volume represents roughly -1024..2000 HU.
   const colors =
     V.Rendering.Core.vtkColorTransferFunction.newInstance();

   colors.addRGBPoint(0,   0.01, 0.01, 0.02);
   colors.addRGBPoint(95,  0.20, 0.27, 0.34);
   colors.addRGBPoint(112, 0.58, 0.64, 0.71);
   colors.addRGBPoint(150, 0.87, 0.85, 0.78);
   colors.addRGBPoint(255, 1.00, 0.98, 0.91);

   const opacity =
     V.Common.DataModel.vtkPiecewiseFunction.newInstance();

   const prop = volume.getProperty();
   prop.setRGBTransferFunction(0, colors);
   prop.setScalarOpacity(0, opacity);
   prop.setInterpolationTypeToLinear();
   prop.setShade(false);
   prop.setScalarOpacityUnitDistance(
     0, Math.max(0.5, Math.min(...spacing) * 2)
   );

   vtkVolumeProperty = prop;

   // Clear any previously attached renderer.
   if (vtkFullScreen) {
     vtkFullScreen.delete();
     vtkFullScreen = null;
   }

   panel.innerHTML = "";

   vtkFullScreen =
     V.Rendering.Misc.vtkFullScreenRenderWindow.newInstance({
       rootContainer: panel,
       containerStyle: {
         position: "absolute",
         width: "100%",
         height: "100%",
         overflow: "hidden"
       },
       background: [0.008, 0.012, 0.025]
     });

   vtkCameraRenderer = vtkFullScreen.getRenderer();
   vtkRenderWindow = vtkFullScreen.getRenderWindow();

   vtkRenderer = vtkCameraRenderer;
   vtkWindow = vtkRenderWindow;

   vtkCameraRenderer.addVolume(volume);
   vtkCameraRenderer.resetCamera();

   function updateOpacity() {
     const value = Number(
       document.getElementById("opacity").value
     ) / 100;

     opacity.removeAllPoints();
     opacity.addPoint(0, 0);
     opacity.addPoint(100, 0);
     opacity.addPoint(111, 0);
     opacity.addPoint(120, 0.04 * value);
     opacity.addPoint(145, 0.23 * value);
     opacity.addPoint(180, 0.52 * value);
     opacity.addPoint(255, 0.78 * value);

     vtkRenderWindow.render();
   }

   document.getElementById("opacity").oninput = updateOpacity;

   document.getElementById("reset3d").onclick = () => {
     vtkCameraRenderer.resetCamera();
     vtkFullScreen.resize();
     vtkRenderWindow.render();
   };

   // Maintain proper WebGL canvas size.
   const observer = new ResizeObserver(() => {
     if (vtkFullScreen) {
       vtkFullScreen.resize();
       vtkRenderWindow.render();
     }
   });

   observer.observe(panel);

   updateOpacity();

   report("3D READY · " + dims.join(" × "));
   status("3D engine ready");

 } catch (err) {
   console.error("[SINUS 3D ERROR]", err);
   const message = String(err.message || err);
   label.textContent = "3D ERROR";

   panel.innerHTML =
     '<div style="padding:65px 20px;color:#ff9f9f;' +
     'font:14px Segoe UI,Arial;line-height:1.8">' +
     '<b>3D initialization failed</b>' +
     '<div id="vtkErrorMessage"></div>' +
     '<p>Check browser F12 → Console.</p></div>';

   document.getElementById("vtkErrorMessage").textContent = message;
   status("3D ERROR: " + message);
 }
}
"""

start_marker = "async function initialize3D(){"
end_marker = 'document.getElementById("preset").onchange='

start = VIEWER_HTML.find(start_marker)
end = VIEWER_HTML.find(end_marker, start)

if start == -1 or end == -1:
    # Support rerunning the patch.
    start_marker = "let vtkFullScreen = null;"
    start = VIEWER_HTML.find(start_marker)
    end = VIEWER_HTML.find(end_marker, start)

if start == -1 or end == -1:
    raise RuntimeError("Could not locate existing 3D initializer.")

VIEWER_HTML = (
    VIEWER_HTML[:start] +
    NEW_3D_JS + "\n" +
    VIEWER_HTML[end:]
)

# ------------------------------------------------------------
# 4. Improve 2D image display
# ------------------------------------------------------------

print("[4/6] Improving CT slice rendering...")

# Preserve native sharp pixels when zooming in.
VIEWER_HTML = VIEWER_HTML.replace(
    'ctx.imageSmoothingEnabled=true;',
    'ctx.imageSmoothingEnabled=(item.zoom<1.5);'
)

# Allow higher DPI canvas rendering.
VIEWER_HTML = VIEWER_HTML.replace(
    'Math.min(window.devicePixelRatio||1,2)',
    'Math.min(window.devicePixelRatio||1,3)'
)

# Add browser error monitoring.
monitor_js = r"""
window.addEventListener("error", event => {
 console.error("Viewer JavaScript error:", event.message);
 const s=document.getElementById("serverStatus");
 if(s) s.textContent="Browser error: "+event.message;
});
window.addEventListener("unhandledrejection", event => {
 console.error("Viewer Promise error:", event.reason);
});
"""

VIEWER_HTML = VIEWER_HTML.replace(
    '</script>\n</body>',
    monitor_js + '\n</script>\n</body>'
)

# ------------------------------------------------------------
# 5. Rotate credentials
# ------------------------------------------------------------

print("[5/6] Rotating previously exposed credentials...")

VIEWER_PASSWORD = secrets.token_urlsafe(24)
VIEWER_SESSION = secrets.token_urlsafe(32)

# Authentication middleware reads these globals at request time.
# Old login cookies will no longer be accepted.

# ------------------------------------------------------------
# 6. Validate running API
# ------------------------------------------------------------

print("[6/6] Validating backend and asset endpoints...")

base = "http://127.0.0.1:8765"

health = requests.get(base + "/health", timeout=10)
health.raise_for_status()

with requests.Session() as session:
    session.cookies.set("sinus_session", VIEWER_SESSION)

    page = session.get(base + "/", timeout=10)
    script = session.get(base + "/vendor/vtk.js", timeout=20)
    volume_response = session.get(
        base + "/api/volume", timeout=30
    )

    assert page.status_code == 200
    assert script.status_code == 200
    assert len(script.content) > 500_000

    expected_size = int(np.prod(
        META["volume3d_shape_zyx"]
    ))

    assert volume_response.status_code == 200
    assert len(volume_response.content) == expected_size

print("\n" + "=" * 68)
print("CELL 4B — PATCH INSTALLED")
print("=" * 68)
print("Backend status       : ONLINE")
print("vtk.js asset         : VERIFIED")
print("3D voxel data       : VERIFIED")
print("2D enhancement      : INSTALLED")
print("Private credentials : ROTATED")
print("Original DICOM      : UNCHANGED")
print()
print("New private password (DO NOT SHARE):")
print(VIEWER_PASSWORD)
print()
print("Existing external HTTPS tunnel can be reused.")
print("Browser rendering still needs testing.")
print("=" * 68)


SINUS VIEWER PRO — 3D ENGINE REPAIR

[1/6] Loading vtk.js browser rendering engine...
vtk.js downloaded successfully.
Bundle size: 2.66 MiB
[2/6] Replacing browser rendering engine...
[3/6] Installing improved 3D volume renderer...
[4/6] Improving CT slice rendering...
[5/6] Rotating previously exposed credentials...
[6/6] Validating backend and asset endpoints...

CELL 4B — PATCH INSTALLED
Backend status       : ONLINE
vtk.js asset         : VERIFIED
3D voxel data       : VERIFIED
2D enhancement      : INSTALLED
Private credentials : ROTATED
Original DICOM      : UNCHANGED

New private password (DO NOT SHARE):
3ci9YMcAiQNEma2yNQhN07sKp36L_Mm0

Existing external HTTPS tunnel can be reused.
Browser rendering still needs testing.


In [ ]:

# ================================================================
# SINUS 3D VIEWER PRO — CELL 4C
# HIGH-QUALITY 3D VOLUME OPTIMIZATION
# SAFE: ORIGINAL DICOM FILES ARE READ-ONLY
# ================================================================

import os
import json
import time
import numpy as np
import SimpleITK as sitk
from pathlib import Path

print("=" * 70)
print("SINUS VIEWER PRO — 3D QUALITY OPTIMIZATION")
print("=" * 70)

required = [
    "CT_IMAGE", "CT_ARRAY", "META",
    "SINUS_VIEWER_RUNTIME", "SINUS_SERVER"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing previous cells: {missing}")

if not SINUS_SERVER.started:
    raise RuntimeError("Viewer backend is not running.")

print("\nAvailable 3D quality levels:")
print("  1 — FAST       (192 max dimension)")
print("  2 — BALANCED   (256 max dimension) [Recommended]")
print("  3 — ULTRA      (320 max dimension)")
print()
print("Higher quality uses more browser GPU memory.")

choice = input("Select quality [2]: ").strip() or "2"

levels = {"1": 192, "2": 256, "3": 320}

if choice not in levels:
    raise ValueError("Enter 1, 2, or 3.")

target = levels[choice]
start = time.time()

# ------------------------------------------------------------
# Validate source geometry
# ------------------------------------------------------------

original_size = CT_IMAGE.GetSize()
original_spacing = CT_IMAGE.GetSpacing()
direction = np.array(
    CT_IMAGE.GetDirection(), dtype=float
).reshape(3, 3)

extent_mm = [
    (n - 1) * s
    for n, s in zip(original_size, original_spacing)
]

print("\n[1/5] Physical CT geometry")
print("Original dimensions XYZ:", original_size)
print("Original spacing XYZ:", original_spacing)
print("Physical extent mm:", tuple(round(x, 2) for x in extent_mm))
print("Direction matrix:")
print(direction)

if min(extent_mm) < 80:
    print("\nWARNING: Unusually small physical scan extent.")
    print("Verify source DICOM PixelSpacing / slice geometry.")
    print("Do not assume physical measurements are correct.")

# ------------------------------------------------------------
# Create resampled volume using SimpleITK
# ------------------------------------------------------------

print("\n[2/5] Resampling CT volume...")

scale = min(1.0, target / max(original_size))

new_size = [
    max(2, round(n * scale))
    for n in original_size
]

# Preserve first and last voxel-center positions.
new_spacing = [
    s * (n - 1) / (m - 1)
    for n, m, s in zip(
        original_size, new_size, original_spacing
    )
]

resampler = sitk.ResampleImageFilter()

resampler.SetSize([int(v) for v in new_size])
resampler.SetOutputSpacing([float(v) for v in new_spacing])
resampler.SetOutputOrigin(CT_IMAGE.GetOrigin())
resampler.SetOutputDirection(CT_IMAGE.GetDirection())
resampler.SetTransform(sitk.Transform())
resampler.SetInterpolator(sitk.sitkLinear)
resampler.SetDefaultPixelValue(-1024)
resampler.SetOutputPixelType(sitk.sitkInt16)
resampler.SetNumberOfThreads(8)

reduced_image = resampler.Execute(CT_IMAGE)
reduced = sitk.GetArrayFromImage(reduced_image)

print("Optimized dimensions ZYX:", reduced.shape)

# ------------------------------------------------------------
# Create browser-ready intensity data
# ------------------------------------------------------------

print("\n[3/5] Encoding browser-friendly volume...")

# Maintain same intensity mapping as previous Cell 3.
normalized = np.clip(
    (reduced.astype(np.float32) + 1024.0)
    * (255.0 / 3024.0),
    0,
    255
).astype(np.uint8)

# ------------------------------------------------------------
# Atomic replacement of temporary runtime files
# ------------------------------------------------------------

print("\n[4/5] Updating temporary viewer volume...")

data_dir = Path(SINUS_VIEWER_RUNTIME["data_dir"])
volume_path = data_dir / "volume3d_u8.npy"
temporary_path = data_dir / "volume3d_pending.npy"

with open(temporary_path, "wb") as f:
    np.save(f, normalized)

# Replace only the temporary cached 3D file.
# Original patient DICOM files are never written.
os.replace(temporary_path, volume_path)

updated_meta = dict(META)
updated_meta["volume3d_shape_zyx"] = [
    int(v) for v in normalized.shape
]
updated_meta["volume3d_spacing_xyz_mm"] = [
    float(v) for v in new_spacing
]

meta_path = data_dir / "metadata.json"
pending_meta = data_dir / "metadata_pending.json"

with open(pending_meta, "w") as f:
    json.dump(updated_meta, f, indent=2)

os.replace(pending_meta, meta_path)

# Existing FastAPI endpoints read this global metadata.
META.update(updated_meta)

SINUS_VIEWER_RUNTIME["volume3d_shape_zyx"] = normalized.shape

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n[5/5] Verifying updated volume...")

loaded = np.load(volume_path, mmap_mode="r")

assert loaded.shape == normalized.shape
assert loaded.dtype == np.uint8
assert tuple(META["volume3d_shape_zyx"]) == loaded.shape

print("\n" + "=" * 70)
print("3D QUALITY OPTIMIZATION COMPLETE")
print("=" * 70)

print("Quality level        :", choice)
print("3D volume shape ZYX  :", loaded.shape)
print("3D data size         :", round(loaded.nbytes / 1048576, 2), "MiB")
print("Spacing XYZ (mm)     :", tuple(round(x, 5) for x in new_spacing))
print("Processing seconds   :", round(time.time() - start, 2))
print("Backend              : PRESERVED")
print("External link        : PRESERVED")
print("Original DICOM files : UNCHANGED")
print()
print("Refresh browser to load new 3D volume.")
print("=" * 70)


SINUS VIEWER PRO — 3D QUALITY OPTIMIZATION

Available 3D quality levels:
  1 — FAST       (192 max dimension)
  2 — BALANCED   (256 max dimension) [Recommended]
  3 — ULTRA      (320 max dimension)

Higher quality uses more browser GPU memory.
Select quality [2]: 2

[1/5] Physical CT geometry
Original dimensions XYZ: (640, 640, 768)
Original spacing XYZ: (0.07999999821186, 0.07999999821186, 0.07999999869621904)
Physical extent mm: (51.12, 51.12, 61.36)
Direction matrix:
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]

Verify source DICOM PixelSpacing / slice geometry.
Do not assume physical measurements are correct.

[2/5] Resampling CT volume...
Optimized dimensions ZYX: (256, 213, 213)

[3/5] Encoding browser-friendly volume...

[4/5] Updating temporary viewer volume...

[5/5] Verifying updated volume...

3D QUALITY OPTIMIZATION COMPLETE
Quality level        : 2
3D volume shape ZYX  : (256, 213, 213)
3D data size         : 11.08 MiB
Spacing XYZ (mm)     : (0.24113, 0.24113, 0.24063)
Processing 

In [ ]:

# ================================================================
# SINUS 3D VIEWER PRO — CELL 4D
# SMOOTH 2D MPR ENGINE V2
# CACHE + PREFETCH + FRAME-SAFE NAVIGATION + SHARP DISPLAY
# ================================================================

import re
import requests

print("=" * 70)
print("SINUS VIEWER — ULTRA-SMOOTH 2D UPGRADE")
print("=" * 70)

if "VIEWER_HTML" not in globals():
    raise RuntimeError("Existing viewer HTML not found.")

if "SINUS_SERVER" not in globals() or not SINUS_SERVER.started:
    raise RuntimeError("Existing viewer server not running.")

print("\n[1/4] Checking existing backend...")

r = requests.get("http://127.0.0.1:8765/health", timeout=10)
r.raise_for_status()

# Everything from draw() up to the 3D initialization globals
# will be replaced. The functional 3D engine stays unchanged.

NEW_2D_ENGINE = r"""
// ============================================================
// SINUS SMOOTH MPR ENGINE V2
// ============================================================

const MPR_CACHE = new Map();
const MPR_PENDING = new Map();

const MPR_CONFIG = {
    maxCachedImages: 180,
    prefetchRadius: 12,
    maxConcurrent: 6,
    wheelSensitivity: 1
};

let mprQueue = [];
let mprActive = 0;
let mprGeneration = 0;

function mprKey(plane, index) {
    return `${plane}:${index}:${level}:${width}`;
}

function cachedImage(key) {
    const image = MPR_CACHE.get(key);
    if (image) {
        MPR_CACHE.delete(key);
        MPR_CACHE.set(key, image);
    }
    return image;
}

function addToCache(key, image) {
    if (MPR_CACHE.has(key)) MPR_CACHE.delete(key);
    MPR_CACHE.set(key, image);

    while (MPR_CACHE.size > MPR_CONFIG.maxCachedImages) {
        const oldest = MPR_CACHE.keys().next().value;
        MPR_CACHE.delete(oldest);
    }
}

function requestSlice(plane, index, priority = false) {
    const key = mprKey(plane, index);

    const cached = cachedImage(key);
    if (cached) return Promise.resolve(cached);

    if (MPR_PENDING.has(key)) {
        return MPR_PENDING.get(key);
    }

    const generation = mprGeneration;

    const promise = new Promise((resolve, reject) => {
        const job = {plane, index, key, generation, resolve, reject};

        if (priority) mprQueue.unshift(job);
        else mprQueue.push(job);

        processQueue();
    });

    MPR_PENDING.set(key, promise);
    return promise;
}

function processQueue() {
    while (mprActive < MPR_CONFIG.maxConcurrent &&
           mprQueue.length > 0) {

        const job = mprQueue.shift();

        if (job.generation !== mprGeneration) {
            MPR_PENDING.delete(job.key);
            job.resolve(null);
            continue;
        }

        mprActive++;

        const img = new Image();
        img.decoding = "async";

        img.onload = () => {
            if (job.generation === mprGeneration) {
                addToCache(job.key, img);
            }

            MPR_PENDING.delete(job.key);
            mprActive--;
            job.resolve(img);
            processQueue();
        };

        img.onerror = () => {
            MPR_PENDING.delete(job.key);
            mprActive--;
            job.reject(new Error("Slice loading failed: " + job.key));
            processQueue();
        };

        img.src = `/api/slice/${job.plane}/${job.index}` +
                  `?level=${level}&width=${width}`;
    }
}

function prefetchAround(plane, index) {
    const s = state[plane];

    // Prioritize neighboring slices.
    for (let distance = 1;
         distance <= MPR_CONFIG.prefetchRadius;
         distance++) {

        const forward = index + distance;
        const backward = index - distance;

        if (forward < s.total) {
            requestSlice(plane, forward).catch(() => {});
        }

        if (backward >= 0) {
            requestSlice(plane, backward).catch(() => {});
        }
    }
}

function draw(plane) {
    const s = state[plane];
    if (!s || !s.img) return;

    const container = document.getElementById("v-" + plane);
    const canvas = document.getElementById("c-" + plane);
    const context = canvas.getContext("2d", {alpha:false});

    const rect = container.getBoundingClientRect();
    const w = rect.width, h = rect.height;

    if (w <= 0 || h <= 0) return;

    const dpr = Math.min(window.devicePixelRatio || 1, 3);

    const cw = Math.round(w * dpr);
    const ch = Math.round(h * dpr);

    if (canvas.width !== cw || canvas.height !== ch) {
        canvas.width = cw;
        canvas.height = ch;
    }

    canvas.style.width = w + "px";
    canvas.style.height = h + "px";

    context.setTransform(dpr,0,0,dpr,0,0);
    context.fillStyle = "#02060c";
    context.fillRect(0,0,w,h);

    const img = s.img;
    const iw = img.naturalWidth;
    const ih = img.naturalHeight;

    const spacing = metadata.spacing;

    const physical = {
        axial: [spacing[0], spacing[1]],
        coronal: [spacing[0], spacing[2]],
        sagittal: [spacing[1], spacing[2]]
    }[plane];

    const physicalWidth = iw * physical[0];
    const physicalHeight = ih * physical[1];

    const fit = Math.min(
        w / physicalWidth,
        h / physicalHeight
    ) * 0.96;

    const drawWidth = physicalWidth * fit * s.zoom;
    const drawHeight = physicalHeight * fit * s.zoom;

    // Smooth downscaling, but avoid synthetic blur when zoomed in.
    const pixelScale = drawWidth / iw;
    context.imageSmoothingEnabled = pixelScale < 1.1;
    context.imageSmoothingQuality = "high";

    const dx = (w - drawWidth) / 2 + s.dx;
    const dy = (h - drawHeight) / 2 + s.dy;

    context.drawImage(img, dx, dy, drawWidth, drawHeight);
}

function loadSlice(plane) {
    const s = state[plane];

    if (!s) return Promise.resolve();

    const index = s.index;
    const seq = ++s.seq;

    const counter = document.getElementById("n-" + plane);
    counter.textContent = `${index + 1} / ${s.total}`;

    const key = mprKey(plane, index);
    const ready = cachedImage(key);

    if (ready) {
        s.img = ready;
        draw(plane);
        prefetchAround(plane, index);
        return Promise.resolve();
    }

    return requestSlice(plane, index, true)
        .then(image => {
            if (seq !== s.seq || !image) return;

            s.img = image;
            draw(plane);
            prefetchAround(plane, index);
        })
        .catch(error => {
            console.error("[MPR]", error);
            status("2D loading error — check connection");
        });
}

function changeSlice(plane, index) {
    const s = state[plane];

    const next = clamp(
        Math.round(index), 0, s.total - 1
    );

    if (next === s.index) return;

    s.index = next;

    document.getElementById("s-" + plane).value = next;

    loadSlice(plane);
}

function initializePlanes() {
    const limits = {
        axial: metadata.shape[0],
        coronal: metadata.shape[1],
        sagittal: metadata.shape[2]
    };

    planes.forEach(plane => {
        const total = limits[plane];

        const s = state[plane] = {
            total,
            index: Math.floor(total/2),
            seq: 0,
            img: null,
            zoom: 1,
            dx: 0,
            dy: 0,
            wheelAccum: 0,
            wheelFrame: 0
        };

        const slider = document.getElementById("s-" + plane);
        const surface = document.getElementById("v-" + plane);

        slider.min = 0;
        slider.max = total - 1;
        slider.step = 1;
        slider.value = s.index;

        slider.oninput = () => {
            changeSlice(plane, Number(slider.value));
        };

        // Smooth controlled mouse-wheel slice navigation.
        surface.addEventListener("wheel", event => {
            event.preventDefault();

            if (event.ctrlKey) {
                const factor = Math.exp(-event.deltaY * 0.0015);

                s.zoom = clamp(
                    s.zoom * factor,
                    0.25,
                    16
                );

                draw(plane);
                return;
            }

            // Normalize mouse versus precision touchpad.
            let delta = event.deltaY;

            if (event.deltaMode === 1) delta *= 16;
            if (event.deltaMode === 2) delta *= surface.clientHeight;

            // Standard mouse notch ~= 100px.
            // Precision touchpads accumulate smaller movements.
            s.wheelAccum += delta / 100;

            if (Math.abs(s.wheelAccum) < 1) {
                return;
            }

            const steps = Math.trunc(s.wheelAccum);
            s.wheelAccum -= steps;

            changeSlice(
                plane,
                s.index + steps * MPR_CONFIG.wheelSensitivity
            );

        }, {passive:false});

        // Pointer drag = pan.
        let dragging = false;
        let previousX = 0;
        let previousY = 0;

        surface.addEventListener("pointerdown", event => {
            dragging = true;
            previousX = event.clientX;
            previousY = event.clientY;

            surface.setPointerCapture(event.pointerId);
        });

        surface.addEventListener("pointermove", event => {
            if (!dragging) return;

            s.dx += event.clientX - previousX;
            s.dy += event.clientY - previousY;

            previousX = event.clientX;
            previousY = event.clientY;

            draw(plane);
        });

        surface.addEventListener("pointerup", () => {
            dragging = false;
        });

        surface.addEventListener("pointercancel", () => {
            dragging = false;
        });

        surface.addEventListener("dblclick", () => {
            s.zoom = 1;
            s.dx = 0;
            s.dy = 0;
            draw(plane);
        });

        loadSlice(plane);
    });
}

// Prevent old cached windows being reused when presets change.
document.addEventListener("change", event => {
    if (event.target && event.target.id === "preset") {
        mprGeneration++;
        mprQueue = [];
        MPR_CACHE.clear();
        MPR_PENDING.clear();
    }
}, true);

"""

print("[2/4] Installing optimized MPR engine...")

start = VIEWER_HTML.find("function draw(plane){")

if start == -1:
    start = VIEWER_HTML.find(
        "// SINUS SMOOTH MPR ENGINE V2"
    )
    if start != -1:
        start = VIEWER_HTML.rfind("// ======", 0, start)

# Cell 4B places this immediately before the 3D functions.
end = VIEWER_HTML.find("let vtkFullScreen = null;", start)

if start == -1 or end == -1 or end <= start:
    raise RuntimeError(
        "Unable to locate 2D engine boundaries. "
        "No changes applied."
    )

VIEWER_HTML = (
    VIEWER_HTML[:start] +
    NEW_2D_ENGINE +
    "\n" +
    VIEWER_HTML[end:]
)

print("[3/4] Checking engine integration...")

required_js = [
    "function draw(plane)",
    "function loadSlice(plane)",
    "function initializePlanes()",
    "function prefetchAround(plane, index)",
    "function initialize3D()",
]

for item in required_js:
    if item not in VIEWER_HTML:
        raise RuntimeError("Missing JS function: " + item)
    print("OK:", item)

# Main FastAPI homepage uses this global HTML dynamically.
assert SINUS_SERVER.started

print("[4/4] Verifying local server...")

r = requests.get(
    "http://127.0.0.1:8765/health",
    timeout=10
)
r.raise_for_status()

print("\n" + "=" * 70)
print("CELL 4D — SMOOTH 2D ENGINE INSTALLED")
print("=" * 70)
print("Backend                 : ONLINE")
print("3D engine               : PRESERVED")
print("Full-resolution 2D      : PRESERVED")
print("Neighbor prefetch       : 12 slices per direction")
print("Image cache             : 180 images maximum")
print("Wheel navigation        : OPTIMIZED")
print("Canvas rendering        : OPTIMIZED")
print("External tunnel         : PRESERVED")
print("Original DICOM files    : UNCHANGED")
print()
print("Refresh the existing browser viewer.")
print("=" * 70)


SINUS VIEWER — ULTRA-SMOOTH 2D UPGRADE

[1/4] Checking existing backend...
[2/4] Installing optimized MPR engine...
[3/4] Checking engine integration...
OK: function draw(plane)
OK: function loadSlice(plane)
OK: function initializePlanes()
OK: function prefetchAround(plane, index)
OK: function initialize3D()
[4/4] Verifying local server...

CELL 4D — SMOOTH 2D ENGINE INSTALLED
Backend                 : ONLINE
3D engine               : PRESERVED
Full-resolution 2D      : PRESERVED
Neighbor prefetch       : 12 slices per direction
Image cache             : 180 images maximum
Wheel navigation        : OPTIMIZED
Canvas rendering        : OPTIMIZED
External tunnel         : PRESERVED
Original DICOM files    : UNCHANGED

Refresh the existing browser viewer.


In [ ]:

# ============================================================
# SINUS 3D VIEWER PRO — CELL 4E
# BROWSER-RESIDENT MPR — NO NETWORK PER SLICE
# PRESERVES EXISTING WORKING 3D ENGINE
# ORIGINAL PATIENT FILES READ-ONLY
# ============================================================

import os
import re
import time
import threading
import numpy as np
import requests
from pathlib import Path
from fastapi import HTTPException
from fastapi.responses import FileResponse

print("=" * 68)
print("SINUS VIEWER — BROWSER-RESIDENT MPR")
print("=" * 68)

needed = ["APP", "VIEWER_HTML", "FULL", "META", "SINUS_SERVER"]
if any(k not in globals() for k in needed):
    raise RuntimeError("Earlier viewer cells are not active.")
if not SINUS_SERVER.started:
    raise RuntimeError("Backend not running.")

# ----------------------------------------------
# 1. Create an 8-bit, native-resolution volume
# ----------------------------------------------
MPR_DIR = Path("/content/sinus_viewer_runtime/mpr")
MPR_DIR.mkdir(parents=True, exist_ok=True)

MPR_PRESETS = {
    (400, 2000): "bone",
    (40, 400): "soft",
    (-600, 1500): "air",
    (300, 1500): "sinus"
}
MPR_LOCK = threading.Lock()

def build_mpr_file(level, width):
    key = (int(level), int(width))
    if key not in MPR_PRESETS:
        raise ValueError("Unsupported preset")

    path = MPR_DIR / (MPR_PRESETS[key] + ".bin")
    if path.exists() and path.stat().st_size == FULL.size:
        return path

    with MPR_LOCK:
        if path.exists() and path.stat().st_size == FULL.size:
            return path

        tmp = path.with_suffix(".pending")
        low = level - width / 2
        scale = 255.0 / width

        out = np.memmap(
            str(tmp), dtype=np.uint8,
            mode="w+", shape=FULL.shape
        )

        # Chunked processing to avoid large temporary arrays.
        for z in range(FULL.shape[0]):
            frame = FULL[z].astype(np.float32)
            frame = np.clip((frame - low) * scale, 0, 255)
            out[z] = frame.astype(np.uint8)

            if z % 128 == 0:
                print(f"  Encoding: {z}/{FULL.shape[0]}")

        out.flush()
        del out
        os.replace(tmp, path)
        return path

print("\n[1/4] Preparing native-resolution bone volume...")
start_time = time.time()
bone_path = build_mpr_file(400, 2000)
print("Prepared:", round(bone_path.stat().st_size/1048576, 1), "MiB")

# ----------------------------------------------
# 2. Protected volume endpoint
# ----------------------------------------------
if not globals().get("SINUS_MPR_ROUTE_READY", False):
    @APP.get("/api/mpr-volume")
    def serve_mpr_volume(level: int = 400, width: int = 2000):
        if (level, width) not in MPR_PRESETS:
            raise HTTPException(400, "Unsupported CT preset")

        p = build_mpr_file(level, width)
        return FileResponse(
            p,
            media_type="application/octet-stream",
            headers={
                "Cache-Control": "no-store",
                "X-Content-Type-Options": "nosniff"
            }
        )

    SINUS_MPR_ROUTE_READY = True

# ----------------------------------------------
# 3. Replace old 2D JS with local-memory engine
# ----------------------------------------------
NEW_MPR_JS = r"""
// ==========================================================
// SINUS BROWSER-RESIDENT MPR ENGINE
// ==========================================================
const MPR = {
  voxels:null,
  loading:false,
  ready:false,
  generation:0,
  raf:{},
  scratch:{}
};

function mprMessage(msg){
  status(msg);
  const el=document.getElementById("scanInfo");
  if(el) el.textContent=msg;
}

async function reloadMPRVolume(){
  const generation=++MPR.generation;
  MPR.ready=false;
  MPR.loading=true;
  MPR.voxels=null;

  const expected=metadata.shape.reduce((a,b)=>a*b,1);
  const url=`/api/mpr-volume?level=${level}&width=${width}`;

  mprMessage("Loading local-memory CT volume...");
  try{
    const res=await fetch(url,{cache:"no-store"});
    if(!res.ok) throw Error("HTTP "+res.status);

    // Fetch once. No server request is needed during scrolling.
    const buffer=await res.arrayBuffer();
    if(generation!==MPR.generation)return;

    if(buffer.byteLength!==expected){
      throw Error(
        "Volume size mismatch: "+buffer.byteLength+
        " vs "+expected
      );
    }

    MPR.voxels=new Uint8Array(buffer);
    MPR.ready=true;
    MPR.loading=false;

    for(const p of planes){
      scheduleMPRDraw(p);
    }
    mprMessage(
      "LOCAL MPR READY · "+
      (buffer.byteLength/1048576).toFixed(1)+
      " MiB · No per-slice network requests"
    );
  }catch(e){
    MPR.loading=false;
    console.error("[LOCAL MPR]",e);
    mprMessage("MPR ERROR: "+e.message);
  }
}

function scheduleMPRDraw(plane){
  if(MPR.raf[plane])return;
  MPR.raf[plane]=requestAnimationFrame(()=>{
    MPR.raf[plane]=null;
    draw(plane);
  });
}

function draw(plane){
  if(!MPR.ready || !MPR.voxels)return;
  const s=state[plane];
  if(!s)return;

  const [Z,Y,X]=metadata.shape;
  const sp=metadata.spacing;
  const volume=MPR.voxels;

  let iw,ih,sx,sy;

  if(plane==="axial"){
    iw=X;ih=Y;sx=sp[0];sy=sp[1];
  }else if(plane==="coronal"){
    iw=X;ih=Z;sx=sp[0];sy=sp[2];
  }else{
    iw=Y;ih=Z;sx=sp[1];sy=sp[2];
  }

  // Reuse scratch buffers. Avoid per-frame allocations.
  let scratch=MPR.scratch[plane];
  if(!scratch || scratch.w!==iw || scratch.h!==ih){
    const off=document.createElement("canvas");
    off.width=iw;off.height=ih;
    const offctx=off.getContext("2d",{alpha:false});
    scratch={
      canvas:off,ctx:offctx,w:iw,h:ih,
      pixels:offctx.createImageData(iw,ih)
    };
    MPR.scratch[plane]=scratch;
  }

  const rgba=scratch.pixels.data;
  const index=s.index;

  // Native-resolution slice extraction.
  // Note: physical patient orientation must be verified separately.
  if(plane==="axial"){
    const base=index*Y*X;
    for(let row=0;row<Y;row++){
      const src=base+(Y-1-row)*X;
      let dst=row*X*4;
      for(let col=0;col<X;col++){
        const v=volume[src+col];
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=255;
      }
    }
  }else if(plane==="coronal"){
    for(let row=0;row<Z;row++){
      const z=Z-1-row;
      const base=(z*Y+index)*X;
      let dst=row*X*4;
      for(let col=0;col<X;col++){
        const v=volume[base+col];
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=255;
      }
    }
  }else{
    for(let row=0;row<Z;row++){
      const z=Z-1-row;
      let dst=row*Y*4;
      for(let col=0;col<Y;col++){
        const v=volume[(z*Y+col)*X+index];
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=255;
      }
    }
  }

  scratch.ctx.putImageData(scratch.pixels,0,0);

  const container=document.getElementById("v-"+plane);
  const canvas=document.getElementById("c-"+plane);
  const ctx=canvas.getContext("2d",{alpha:false});

  const w=container.clientWidth;
  const h=container.clientHeight;
  if(!w || !h)return;

  const dpr=Math.min(window.devicePixelRatio||1,2);
  const cw=Math.round(w*dpr);
  const ch=Math.round(h*dpr);

  if(canvas.width!==cw || canvas.height!==ch){
    canvas.width=cw;
    canvas.height=ch;
  }

  canvas.style.width=w+"px";
  canvas.style.height=h+"px";
  ctx.setTransform(dpr,0,0,dpr,0,0);
  ctx.fillStyle="#02060c";
  ctx.fillRect(0,0,w,h);

  const physicalW=iw*sx;
  const physicalH=ih*sy;
  const fit=Math.min(w/physicalW,h/physicalH)*0.97;
  const dw=physicalW*fit*s.zoom;
  const dh=physicalH*fit*s.zoom;

  // High-quality downscaling; native pixels on zoom-in.
  ctx.imageSmoothingEnabled=(dw/iw<1);
  ctx.imageSmoothingQuality="high";

  ctx.drawImage(
    scratch.canvas,
    (w-dw)/2+s.dx,
    (h-dh)/2+s.dy,
    dw,dh
  );

  document.getElementById("n-"+plane).textContent=
    `${s.index+1} / ${s.total}`;
}

function loadSlice(plane){
  scheduleMPRDraw(plane);
}

function initializePlanes(){
  const limits={
    axial:metadata.shape[0],
    coronal:metadata.shape[1],
    sagittal:metadata.shape[2]
  };

  planes.forEach(plane=>{
    const total=limits[plane];
    const s=state[plane]={
      index:Math.floor(total/2),
      total,zoom:1,dx:0,dy:0,wheel:0
    };

    const slider=document.getElementById("s-"+plane);
    const surface=document.getElementById("v-"+plane);
    slider.min=0;slider.max=total-1;
    slider.value=s.index;

    const moveTo=i=>{
      const n=clamp(Math.round(i),0,total-1);
      if(n===s.index)return;
      s.index=n;
      slider.value=n;
      scheduleMPRDraw(plane);
    };

    slider.oninput=()=>moveTo(Number(slider.value));

    surface.addEventListener("wheel",e=>{
      e.preventDefault();

      let delta=e.deltaY;
      if(e.deltaMode===1)delta*=16;
      if(e.deltaMode===2)delta*=surface.clientHeight;

      if(e.ctrlKey){
        s.zoom=clamp(
          s.zoom*Math.exp(-delta*.0015),
          .25,16
        );
        scheduleMPRDraw(plane);
        return;
      }

      s.wheel+=delta;
      // Typical mouse notch ~100; touchpad fractional input
      // accumulates before advancing a slice.
      const steps=Math.trunc(s.wheel/70);
      if(steps){
        s.wheel-=steps*70;
        moveTo(s.index+steps);
      }
    },{passive:false});

    let drag=null;
    surface.addEventListener("pointerdown",e=>{
      drag=[e.clientX,e.clientY];
      surface.setPointerCapture(e.pointerId);
    });
    surface.addEventListener("pointermove",e=>{
      if(!drag)return;
      s.dx+=e.clientX-drag[0];
      s.dy+=e.clientY-drag[1];
      drag=[e.clientX,e.clientY];
      scheduleMPRDraw(plane);
    });
    const stop=()=>drag=null;
    surface.addEventListener("pointerup",stop);
    surface.addEventListener("pointercancel",stop);
    surface.addEventListener("dblclick",()=>{
      s.zoom=1;s.dx=0;s.dy=0;
      scheduleMPRDraw(plane);
    });
  });

  reloadMPRVolume();
}
"""

print("\n[2/4] Installing new browser MPR engine...")

old_marker = "// SINUS SMOOTH MPR ENGINE V2"
new_marker = "// SINUS BROWSER-RESIDENT MPR ENGINE"

start = VIEWER_HTML.find(old_marker)
if start == -1:
    start = VIEWER_HTML.find(new_marker)

end = VIEWER_HTML.find("let vtkFullScreen = null;")

if start == -1 or end == -1 or end <= start:
    raise RuntimeError(
        "Cannot identify engine boundary. HTML unchanged."
    )

# Preserve JavaScript before and after 2D functions,
# including the working 3D renderer.
VIEWER_HTML = (
    VIEWER_HTML[:start] +
    NEW_MPR_JS + "\n" +
    VIEWER_HTML[end:]
)

# Presets now trigger a full windowed volume refresh,
# not individual PNG requests.
VIEWER_HTML = VIEWER_HTML.replace(
    "planes.forEach(loadSlice);",
    "reloadMPRVolume();"
)

print("[3/4] Verifying engine...")

assert "function initialize3D()" in VIEWER_HTML
assert "function initializePlanes()" in VIEWER_HTML
assert "async function reloadMPRVolume()" in VIEWER_HTML
assert 'src="/vendor/vtk.js"' in VIEWER_HTML

print("2D local-memory engine: INSTALLED")
print("3D vtk.js engine: PRESERVED")

print("\n[4/4] Checking backend...")

r=requests.get(
    "http://127.0.0.1:8765/health",
    timeout=10
)
r.raise_for_status()

print("\n" + "=" * 68)
print("CELL 4E COMPLETE")
print("=" * 68)
print("Backend            : ONLINE")
print("Original resolution:", FULL.shape)
print("MPR volume         :", round(bone_path.stat().st_size/1048576,1),"MiB")
print("Per-slice requests : DISABLED after volume loads")
print("3D viewer         : PRESERVED")
print("External tunnel    : PRESERVED")
print("Original DICOM     : UNCHANGED")
print("Preparation time   :", round(time.time()-start_time,1),"seconds")
print()
print("Refresh browser using Ctrl+Shift+R.")
print("Wait for LOCAL MPR READY before testing scrolling.")
print("=" * 68)


SINUS VIEWER — BROWSER-RESIDENT MPR

[1/4] Preparing native-resolution bone volume...
  Encoding: 0/768
  Encoding: 128/768
  Encoding: 256/768
  Encoding: 384/768
  Encoding: 512/768
  Encoding: 640/768
Prepared: 300.0 MiB

[2/4] Installing new browser MPR engine...
[3/4] Verifying engine...
2D local-memory engine: INSTALLED
3D vtk.js engine: PRESERVED

[4/4] Checking backend...

CELL 4E COMPLETE
Backend            : ONLINE
Original resolution: (768, 640, 640)
MPR volume         : 300.0 MiB
Per-slice requests : DISABLED after volume loads
3D viewer         : PRESERVED
External tunnel    : PRESERVED
Original DICOM     : UNCHANGED
Preparation time   : 2.2 seconds

Refresh browser using Ctrl+Shift+R.
Wait for LOCAL MPR READY before testing scrolling.


In [ ]:

# ============================================================
# SINUS VIEWER PRO — CELL 4F
# COMPRESSED BROWSER VOLUME DELIVERY
# KEEP SAME SMOOTH CELL 4E VIEWER
# ============================================================

import os
import time
import zlib
from pathlib import Path
from fastapi import HTTPException
from fastapi.responses import FileResponse
from fastapi.routing import APIRoute

print("=" * 65)
print("SINUS VIEWER — DOWNLOAD OPTIMIZATION")
print("=" * 65)

assert "APP" in globals(), "Cell 4 required"
assert "build_mpr_file" in globals(), "Cell 4E required"
assert SINUS_SERVER.started, "Server not running"

# Confirm existing MPR endpoint.
old_routes = [
    r for r in APP.router.routes
    if isinstance(r, APIRoute)
    and r.path == "/api/mpr-volume"
]

if not old_routes:
    raise RuntimeError("Cell 4E endpoint not found.")

COMPRESS_DIR = Path("/content/sinus_viewer_runtime/mpr")
COMPRESS_DIR.mkdir(parents=True, exist_ok=True)

def prepare_compressed(path):
    source = Path(path)
    target = Path(str(source) + ".gz")

    if (target.exists()
        and target.stat().st_mtime >= source.stat().st_mtime):
        return target

    temp = Path(str(target) + ".pending")

    try:
        compressor = zlib.compressobj(
            level=1,
            wbits=16 + zlib.MAX_WBITS
        )

        with open(source, "rb") as inp, open(temp, "wb") as out:
            while True:
                block = inp.read(4 * 1024 * 1024)
                if not block:
                    break
                out.write(compressor.compress(block))

            out.write(compressor.flush())

        os.replace(temp, target)
        return target
    finally:
        if temp.exists():
            temp.unlink()

print("\n[1/3] Testing compression on Bone preset...")
t0 = time.time()

source = build_mpr_file(400, 2000)
compressed = prepare_compressed(source)

original_mb = source.stat().st_size / 1048576
compressed_mb = compressed.stat().st_size / 1048576

ratio = compressed_mb / original_mb
USE_COMPRESSION = ratio < 0.90

print("Original volume :", round(original_mb, 2), "MiB")
print("Compressed      :", round(compressed_mb, 2), "MiB")
print("Size reduction  :", round((1-ratio)*100, 1), "%")
print("Preparation     :", round(time.time()-t0, 2), "s")

# Restore original route if the compression patch is rerun.
for route in old_routes:
    APP.router.routes.remove(route)

print("\n[2/3] Installing optimized endpoint...")

@APP.get("/api/mpr-volume")
def optimized_mpr_volume(level: int = 400, width: int = 2000):
    if (level, width) not in MPR_PRESETS:
        raise HTTPException(400, "Unsupported CT preset")

    original = build_mpr_file(level, width)

    if USE_COMPRESSION:
        packed = prepare_compressed(original)
        return FileResponse(
            packed,
            media_type="application/octet-stream",
            headers={
                "Content-Encoding": "gzip",
                "Cache-Control": "private, no-store",
                "X-Content-Type-Options": "nosniff"
            }
        )

    return FileResponse(
        original,
        media_type="application/octet-stream",
        headers={
            "Cache-Control": "private, no-store",
            "X-Content-Type-Options": "nosniff"
        }
    )

print("[3/3] Checking endpoint configuration...")

assert any(
    isinstance(r, APIRoute)
    and r.path == "/api/mpr-volume"
    for r in APP.router.routes
)

print("\n" + "=" * 65)
print("CELL 4F COMPLETE")
print("=" * 65)
print("Delivery mode       :", "GZIP" if USE_COMPRESSION else "RAW")
print("Transfer reduction  :", round((1-ratio)*100, 1), "%")
print("3D engine           : UNCHANGED")
print("2D resolution       : UNCHANGED")
print("Existing tunnel     : UNCHANGED")
print("Original DICOM      : UNCHANGED")
print()
print("Refresh browser to test download performance.")
print("=" * 65)


SINUS VIEWER — DOWNLOAD OPTIMIZATION

[1/3] Testing compression on Bone preset...
Original volume : 300.0 MiB
Compressed      : 195.82 MiB
Size reduction  : 34.7 %
Preparation     : 8.79 s

[2/3] Installing optimized endpoint...
[3/3] Checking endpoint configuration...

CELL 4F COMPLETE
Delivery mode       : GZIP
Transfer reduction  : 34.7 %
3D engine           : UNCHANGED
2D resolution       : UNCHANGED
Existing tunnel     : UNCHANGED
Original DICOM      : UNCHANGED

Refresh browser to test download performance.


In [ ]:

# ============================================================
# SINUS 3D VIEWER PRO — CELL 5
# TEMPORARY HTTPS LINK + SESSION SECURITY
# ============================================================

import os
import re
import time
import secrets
import hmac
import subprocess
import urllib.request
from pathlib import Path

import requests
from IPython.display import display, HTML

print("=" * 68)
print("SINUS 3D VIEWER PRO — EXTERNAL BROWSER LAUNCHER")
print("=" * 68)

# 1. Verify Colab backend
if "SINUS_SERVER" not in globals() or not SINUS_SERVER.started:
    raise RuntimeError("Cell 4 backend not running.")

local_url = "http://127.0.0.1:8765"

response = requests.get(
    local_url + "/health",
    timeout=10
)
response.raise_for_status()

print("\n[1/5] Backend verified:", response.json())

# 2. Rotate previous credentials
VIEWER_PASSWORD = secrets.token_urlsafe(24)
VIEWER_SESSION = secrets.token_urlsafe(32)

print("[2/5] New private session credentials generated.")
print("Previous password and login cookies invalidated.")

# 3. Download Cloudflare's official tunnel client
binary = Path("/content/cloudflared")

if not binary.exists():
    print("[3/5] Downloading Cloudflare tunnel client...")

    url = (
        "https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/cloudflared-linux-amd64"
    )

    urllib.request.urlretrieve(url, str(binary))
    binary.chmod(0o755)
else:
    print("[3/5] Existing Cloudflare client found.")

version = subprocess.run(
    [str(binary), "--version"],
    capture_output=True,
    text=True,
    timeout=15,
    check=True
)
print(version.stdout.strip())

# 4. Stop previous tunnel if any
if "SINUS_TUNNEL" in globals():
    if SINUS_TUNNEL.poll() is None:
        SINUS_TUNNEL.terminate()
        try:
            SINUS_TUNNEL.wait(timeout=8)
        except subprocess.TimeoutExpired:
            SINUS_TUNNEL.kill()
            SINUS_TUNNEL.wait(timeout=5)

print("[4/5] Starting temporary HTTPS tunnel...")

log_path = Path("/content/sinus_viewer_tunnel.log")

log_handle = open(log_path, "w", encoding="utf-8")

SINUS_TUNNEL = subprocess.Popen(
    [
        str(binary),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--url", local_url
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

log_handle.close()

# Discover actual URL from tunnel logs
public_url = None
pattern = re.compile(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
)

for attempt in range(35):
    if SINUS_TUNNEL.poll() is not None:
        break

    contents = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    match = pattern.search(contents)

    if match:
        public_url = match.group(0)
        break

    time.sleep(1)

if not public_url:
    print("\nTunnel log (last lines):")
    print("\n".join(
        log_path.read_text(errors="replace").splitlines()[-15:]
    ))
    raise RuntimeError(
        "External URL not generated. Tunnel may be unavailable "
        "or blocked by this runtime."
    )

SINUS_PUBLIC_URL = public_url

# 5. Test external endpoint without authenticating
print("[5/5] Checking external HTTPS endpoint...")

external_status = "NOT VERIFIED"

for attempt in range(8):
    try:
        r = requests.get(
            public_url + "/login",
            timeout=12,
            allow_redirects=False
        )

        if r.status_code == 200 and "Sinus Viewer Login" in r.text:
            external_status = "VERIFIED"
            break

    except requests.RequestException:
        pass

    time.sleep(2)

print("\n" + "=" * 68)
print("SINUS 3D VIEWER — EXTERNAL ACCESS")
print("=" * 68)
print("Backend         : ONLINE")
print("Temporary link  :", SINUS_PUBLIC_URL)
print("HTTPS status    :", external_status)
print("Patient files   : UNCHANGED")
print()
print("NEW PRIVATE PASSWORD:")
print(VIEWER_PASSWORD)
print()
print("Keep this password private.")
print("Do not share screenshots containing the password.")
print("=" * 68)

if external_status == "VERIFIED":
    display(HTML(
        '<div style="padding:15px;background:#102235;'
        'border:1px solid #387fc0;border-radius:12px">'
        '<h3 style="color:#e9f4ff">Sinus Viewer Ready</h3>'
        f'<a href="{SINUS_PUBLIC_URL}" target="_blank" '
        'rel="noopener noreferrer" '
        'style="color:#72b7ff;font-size:16px">'
        'OPEN SINUS 3D VIEWER ↗</a>'
        '</div>'
    ))
else:
    print(
        "Tunnel URL generated, but external access could not "
        "be verified. Do not assume it is working yet."
    )

print("\nCELL 5 COMPLETE")


SINUS 3D VIEWER PRO — EXTERNAL BROWSER LAUNCHER

[1/5] Backend verified: {'status': 'running'}
[2/5] New private session credentials generated.
Previous password and login cookies invalidated.
[3/5] Downloading Cloudflare tunnel client...
cloudflared version 2026.10.1 (built 2026-10-15-17:08 UTC)
[4/5] Starting temporary HTTPS tunnel...
[5/5] Checking external HTTPS endpoint...

SINUS 3D VIEWER — EXTERNAL ACCESS
Backend         : ONLINE
Temporary link  : https://investment-sensors-opinion-redhead.trycloudflare.com
HTTPS status    : NOT VERIFIED
Patient files   : UNCHANGED

NEW PRIVATE PASSWORD:
BCbm-XRQ0jB0YoBpC9fXJamfWp9zpjuF

Keep this password private.
Do not share screenshots containing the password.
Tunnel URL generated, but external access could not be verified. Do not assume it is working yet.

CELL 5 COMPLETE


In [ ]:

import requests
from pathlib import Path

print("=" * 60)
print("SINUS VIEWER — CONNECTION DIAGNOSTIC")
print("=" * 60)

# Local backend
try:
    r = requests.get(
        "http://127.0.0.1:8765/health",
        timeout=10
    )
    print("LOCAL BACKEND:", r.status_code, r.text[:100])
except Exception as e:
    print("LOCAL ERROR:", str(e)[:200])

# Cloudflare tunnel process
if "SINUS_TUNNEL" in globals():
    print(
        "TUNNEL:",
        "RUNNING" if SINUS_TUNNEL.poll() is None
        else f"STOPPED ({SINUS_TUNNEL.returncode})"
    )
else:
    print("TUNNEL: NOT FOUND")

# External HTTPS
if "SINUS_PUBLIC_URL" in globals():
    for endpoint in ["/health", "/login"]:
        try:
            r = requests.get(
                SINUS_PUBLIC_URL + endpoint,
                timeout=15,
                allow_redirects=False
            )
            print("EXTERNAL", endpoint, ":", r.status_code)
            print("Content-Type:", r.headers.get("content-type"))
            if endpoint == "/login":
                print(
                    "Login page:",
                    "Sinus Viewer Login" in r.text
                )
        except Exception as e:
            print("EXTERNAL", endpoint, "ERROR:", str(e)[:200])

# Tunnel log
log = Path("/content/sinus_viewer_tunnel.log")
if log.exists():
    print("\nLAST TUNNEL LOG LINES:")
    for line in log.read_text(errors="replace").splitlines()[-12:]:
        # Avoid printing URLs or possible credentials
        if "https://" in line or "token" in line.lower():
            continue
        print(line[:250])

print("=" * 60)
print("DIAGNOSTIC COMPLETE")
print("=" * 60)


SINUS VIEWER — CONNECTION DIAGNOSTIC
LOCAL BACKEND: 200 {"status":"running"}
TUNNEL: RUNNING
EXTERNAL /health : 200
Content-Type: application/json
EXTERNAL /login : 200
Content-Type: text/html; charset=utf-8
Login page: True

LAST TUNNEL LOG LINES:
2026-10-20T08:59:34Z INF |  Cloudflare API    api.cloudflare.com:443     PASS    API is reachable               |
2026-10-20T08:59:34Z INF |                                                                                      |
2026-10-20T08:59:34Z INF |  SUMMARY: Environment is healthy. cloudflared will use 'http2' as primary protocol.  |
2026-10-20T08:59:34Z INF +--------------------------------------------------------------------------------------+
2026-10-20T08:59:34Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=61441383-8caf-4221-ac84-ef2f173f914d status=pass target=region1.v2.argotunnel.com
2026-10-20T08:59:34Z INF precheck component="DNS Resolution" details="DNS Resolved successfully" run_id=61441

In [ ]:

import secrets

VIEWER_PASSWORD = secrets.token_urlsafe(24)
VIEWER_SESSION = secrets.token_urlsafe(32)

print("New credentials generated successfully.")
print("Previous password and sessions invalidated.")
print("New private password (do not share):")
print(VIEWER_PASSWORD)


New credentials generated successfully.
Previous password and sessions invalidated.
New private password (do not share):
XHdjNNgcsOb4GrZ-6xAyH7q3e9Wo3The


In [4]:
# SINUS 3D VIEWER PRO — ONE-CELL COLAB LAUNCHER
# Based on the uploaded working notebook. DICOM source is read-only.
# Re-running this cell replaces ONLY temporary files in /content/sinus_viewer_runtime.
# Private medical data: use only with authorization and never share the viewer URL/password.

import os, time, secrets
if "SINUS_TUNNEL" in globals():
    try:
        if SINUS_TUNNEL.poll() is None:
            SINUS_TUNNEL.terminate()
            SINUS_TUNNEL.wait(timeout=8)
    except Exception:
        try: SINUS_TUNNEL.kill()
        except Exception: pass
if "SINUS_SERVER" in globals():
    try:
        SINUS_SERVER.should_exit = True
        if "SINUS_SERVER_THREAD" in globals():
            SINUS_SERVER_THREAD.join(timeout=10)
    except Exception: pass
# Reset idempotence flags: FastAPI app is recreated below.
for _name in ("VTK_ROUTE_INSTALLED", "SINUS_MPR_ROUTE_READY"):
    globals().pop(_name, None)
# Extra dependencies for form login and compressed streaming.
import sys, subprocess
subprocess.check_call([sys.executable,"-m","pip","install","-q", "python-multipart", "Pillow", "requests"])


# ===================== FROM VERIFIED SOURCE CELL 1 =====================

# ============================================================
# SINUS 3D VIEWER PRO — CELL 1
# ENVIRONMENT SETUP + GOOGLE DRIVE
# ============================================================

import os
import sys
import subprocess
import platform

print("=" * 65)
print("SINUS 3D VIEWER PRO — INITIAL SETUP")
print("=" * 65)

packages = [
    "SimpleITK",
    "pydicom",
    "numpy",
    "scipy",
    "fastapi",
    "uvicorn",
    "orjson",
    "zstandard",
    "psutil"
]

subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "-q", *packages
])

# from google.colab import drive
# drive.mount("/content/drive")

import torch
import psutil
import SimpleITK as sitk
import pydicom
import fastapi
import uvicorn

print("\nSYSTEM INFORMATION")
print("-" * 65)
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3,
            2
        ),
        "GB"
    )

print(
    "System RAM:",
    round(psutil.virtual_memory().total / 1024**3, 2),
    "GB"
)

print("SimpleITK:", sitk.Version_VersionString())

print("\nGOOGLE DRIVE")
print("-" * 65)
print("Mounted:", os.path.isdir("/content/drive/MyDrive"))

print("\nORIGINAL DATASET SAFETY")
print("Read-only inspection policy: ENABLED")
print("No patient files modified.")

print("\n" + "=" * 65)
print("CELL 1 COMPLETE")
print("=" * 65)



# ===================== FROM VERIFIED SOURCE CELL 2 =====================

# ============================================================
# SINUS 3D VIEWER PRO — CELL 2
# SMART DICOM SCANNER + SERIES SELECTOR
# STRICTLY READ-ONLY
# ============================================================

import os
import time
from pathlib import Path

import SimpleITK as sitk
import numpy as np

print("=" * 70)
print("SINUS 3D VIEWER PRO — PATIENT DICOM SCANNER")
print("=" * 70)

print("\nPaste patient folder FULL PATH.")
print("Example: /content/drive/MyDrive/Sinus_Project/Sinus/PATIENT_FOLDER")

raw_path = input("\nPATIENT FOLDER PATH: ").strip().strip('"').strip("'")
PATIENT_PATH = Path(raw_path).expanduser()

if not PATIENT_PATH.is_dir():
    raise FileNotFoundError(
        f"Patient folder not found: {PATIENT_PATH}"
    )

print("\n[1/5] Scanning folders...")
start_time = time.time()

DICOM_SERIES = []
scanned_folders = 0
scan_errors = []

for root, dirs, files in os.walk(PATIENT_PATH):
    dirs.sort()

    if not files:
        continue

    scanned_folders += 1

    try:
        ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(root)

        if not ids:
            continue

        for series_id in ids:
            paths = list(
                sitk.ImageSeriesReader.GetGDCMSeriesFileNames(
                    root, series_id
                )
            )

            if not paths:
                continue

            DICOM_SERIES.append({
                "folder": root,
                "series_id": series_id,
                "files": paths,
                "count": len(paths)
            })

    except Exception as exc:
        scan_errors.append((root, str(exc)[:150]))

if not DICOM_SERIES:
    raise RuntimeError(
        "No readable DICOM image series found. "
        "Check the folder path and DICOM files."
    )

DICOM_SERIES.sort(
    key=lambda item: item["count"],
    reverse=True
)

print("\n[2/5] Available DICOM series")
print("-" * 70)

for index, item in enumerate(DICOM_SERIES):
    folder_name = Path(item["folder"]).name

    print(
        f"[{index}] "
        f"{item['count']} files | "
        f"Folder: {folder_name}"
    )

print("-" * 70)
print("Total series:", len(DICOM_SERIES))
print("Folders scanned:", scanned_folders)
print("Folders with scan errors:", len(scan_errors))

print("\n[3/5] Series selection")
print("Default = 0 (series with most files)")

choice = (input("Select series number [0] (Enter for default): ").strip()
          if len(DICOM_SERIES) > 1 else "0")

try:
    selected_index = int(choice) if choice else 0
except ValueError:
    raise ValueError("Series selection must be a number.")

if not 0 <= selected_index < len(DICOM_SERIES):
    raise ValueError("Invalid series number.")

SELECTED_DICOM_SERIES = DICOM_SERIES[selected_index]

print("\n[4/5] Loading selected series into RAM...")

reader = sitk.ImageSeriesReader()
reader.SetFileNames(SELECTED_DICOM_SERIES["files"])

reader.MetaDataDictionaryArrayUpdateOn()
reader.LoadPrivateTagsOff()

CT_IMAGE = reader.Execute()

# SimpleITK image = X,Y,Z
# NumPy image = Z,Y,X
CT_ARRAY = sitk.GetArrayFromImage(CT_IMAGE)

CT_SIZE = CT_IMAGE.GetSize()
CT_SPACING = CT_IMAGE.GetSpacing()
CT_ORIGIN = CT_IMAGE.GetOrigin()
CT_DIRECTION = CT_IMAGE.GetDirection()

print("\n[5/5] Validating volume...")

if CT_ARRAY.ndim != 3:
    raise RuntimeError(
        f"Expected 3D volume, received {CT_ARRAY.ndim}D."
    )

if any(value <= 0 for value in CT_SPACING):
    raise RuntimeError("Invalid voxel spacing.")

finite_mask = np.isfinite(CT_ARRAY)

if not finite_mask.all():
    raise RuntimeError("Non-finite voxel values detected.")

# Check for orthonormal physical direction matrix.
direction_matrix = np.array(CT_DIRECTION).reshape(3, 3)
orthogonal_error = np.max(
    np.abs(direction_matrix.T @ direction_matrix - np.eye(3))
)

if orthogonal_error > 0.01:
    print("WARNING: Unusual image direction matrix.")

# Store global viewer-ready context.
SINUS_VIEWER_CONTEXT = {
    "patient_folder": str(PATIENT_PATH),
    "series_index": selected_index,
    "file_count": SELECTED_DICOM_SERIES["count"],
    "shape_zyx": CT_ARRAY.shape,
    "spacing_xyz": CT_SPACING,
    "origin_xyz": CT_ORIGIN,
    "direction": CT_DIRECTION,
    "volume_loaded": True
}

memory_gib = CT_ARRAY.nbytes / (1024 ** 3)
scan_seconds = time.time() - start_time

print("\n" + "=" * 70)
print("DICOM VOLUME REPORT")
print("=" * 70)

print("Selected series       :", selected_index)
print("DICOM files           :", SELECTED_DICOM_SERIES["count"])
print("Volume dimensions XYZ :", CT_SIZE)
print("NumPy dimensions ZYX  :", CT_ARRAY.shape)
print("Voxel spacing XYZ (mm):", tuple(round(x, 4) for x in CT_SPACING))
print("Data type             :", CT_ARRAY.dtype)
print("Minimum intensity     :", float(CT_ARRAY.min()))
print("Maximum intensity     :", float(CT_ARRAY.max()))
print("Volume RAM usage      :", round(memory_gib, 3), "GiB")
print("Scan + load time      :", round(scan_seconds, 2), "seconds")

if CT_ARRAY.shape[0] < 2:
    print("\nWARNING: Only one slice detected.")
    print("Full 3D reconstruction requires multiple slices.")

if len(DICOM_SERIES) > 1:
    print("\nNOTE: Multiple series found.")
    print("Verify that you selected the intended CT acquisition.")

print("\nORIGINAL DATASET SAFETY")
print("Source files modified : NO")
print("Files renamed         : NO")
print("Files moved           : NO")
print("Files deleted         : NO")

print("\n" + "=" * 70)
print("CELL 2 COMPLETE — VOLUME LOADED INTO RAM")
print("=" * 70)



# ===================== FROM VERIFIED SOURCE CELL 3 =====================
# ============================================================
# SINUS 3D VIEWER PRO — CELL 3
# RUNTIME PREPARATION + OPTIMIZED VIEWER DATA
# STRICTLY READ-ONLY
# ============================================================

import os
import io
import json
import math
import shutil
from pathlib import Path

import numpy as np
import SimpleITK as sitk
from scipy.ndimage import zoom

print("=" * 72)
print("SINUS 3D VIEWER PRO — RUNTIME PREPARATION")
print("=" * 72)

# -----------------------------
# Required globals check
# -----------------------------
required_globals = ["CT_ARRAY", "CT_IMAGE", "SINUS_VIEWER_CONTEXT"]
missing = [name for name in required_globals if name not in globals()]
if missing:
    raise RuntimeError(
        f"Missing required objects from Cell 2: {missing}"
    )

# -----------------------------
# Runtime folder
# -----------------------------
RUNTIME_DIR = Path("/content/sinus_viewer_runtime")
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = RUNTIME_DIR / "data"
STATIC_DIR = RUNTIME_DIR / "static"
DATA_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

print("\n[1/7] Copying volume into viewer runtime...")
volume_i16 = np.asarray(CT_ARRAY, dtype=np.int16)   # Z,Y,X
zdim, ydim, xdim = volume_i16.shape

spacing_xyz = tuple(float(x) for x in CT_IMAGE.GetSpacing())
origin_xyz = tuple(float(x) for x in CT_IMAGE.GetOrigin())
direction = tuple(float(x) for x in CT_IMAGE.GetDirection())

# -----------------------------
# HU window helper
# -----------------------------
def window_to_uint8(arr, level, width):
    low = level - width / 2.0
    high = level + width / 2.0
    clipped = np.clip(arr, low, high)
    scaled = ((clipped - low) / (high - low) * 255.0).astype(np.uint8)
    return scaled

# -----------------------------
# Thumbnail previews
# -----------------------------
print("[2/7] Generating quick preview thumbnails...")

mid_z = zdim // 2
mid_y = ydim // 2
mid_x = xdim // 2

preview_axial = window_to_uint8(volume_i16[mid_z], 400, 2000)
preview_coronal = window_to_uint8(volume_i16[:, mid_y, :], 400, 2000)
preview_sagittal = window_to_uint8(volume_i16[:, :, mid_x], 400, 2000)

np.save(DATA_DIR / "preview_axial.npy", preview_axial)
np.save(DATA_DIR / "preview_coronal.npy", preview_coronal)
np.save(DATA_DIR / "preview_sagittal.npy", preview_sagittal)

# -----------------------------
# 3D volume downsampling
# -----------------------------
print("[3/7] Preparing optimized 3D volume...")

# Target maximum side for browser-friendly 3D
target_max_side = 192

scale_z = min(1.0, target_max_side / zdim)
scale_y = min(1.0, target_max_side / ydim)
scale_x = min(1.0, target_max_side / xdim)

scale = min(scale_z, scale_y, scale_x)

if scale < 1.0:
    vol_small = zoom(
        volume_i16.astype(np.float32),
        zoom=(scale, scale, scale),
        order=1
    ).astype(np.float32)
else:
    vol_small = volume_i16.astype(np.float32)

# Bone-friendly normalization for volume rendering
# We keep air/bone visibility balanced
vol_small_clip = np.clip(vol_small, -1024, 2000)
vol_small_u8 = ((vol_small_clip + 1024.0) / (2000.0 + 1024.0) * 255.0)
vol_small_u8 = np.clip(vol_small_u8, 0, 255).astype(np.uint8)

np.save(DATA_DIR / "volume3d_u8.npy", vol_small_u8)

scaled_spacing_xyz = (
    spacing_xyz[0] * (xdim / vol_small_u8.shape[2]),
    spacing_xyz[1] * (ydim / vol_small_u8.shape[1]),
    spacing_xyz[2] * (zdim / vol_small_u8.shape[0]),
)

# -----------------------------
# Save main volume for slice service
# -----------------------------
print("[4/7] Saving full-resolution volume for slice streaming...")
np.save(DATA_DIR / "volume_full_i16.npy", volume_i16)

# -----------------------------
# Metadata
# -----------------------------
print("[5/7] Writing viewer metadata...")

viewer_metadata = {
    "patient_folder": SINUS_VIEWER_CONTEXT["patient_folder"],
    "series_index": int(SINUS_VIEWER_CONTEXT["series_index"]),
    "file_count": int(SINUS_VIEWER_CONTEXT["file_count"]),
    "shape_zyx": [int(zdim), int(ydim), int(xdim)],
    "spacing_xyz_mm": [float(x) for x in spacing_xyz],
    "origin_xyz": [float(x) for x in origin_xyz],
    "direction": [float(x) for x in direction],
    "dtype": str(volume_i16.dtype),
    "hu_min": int(volume_i16.min()),
    "hu_max": int(volume_i16.max()),
    "default_window": {
        "bone": {"level": 400, "width": 2000},
        "sinus": {"level": 300, "width": 1500},
        "soft_tissue": {"level": 40, "width": 400},
        "air": {"level": -600, "width": 1500},
    },
    "mid_slices": {
        "axial": int(mid_z),
        "coronal": int(mid_y),
        "sagittal": int(mid_x),
    },
    "volume3d_shape_zyx": [int(v) for v in vol_small_u8.shape],
    "volume3d_spacing_xyz_mm": [float(x) for x in scaled_spacing_xyz],
    "runtime_dir": str(RUNTIME_DIR),
    "notes": [
        "Original DICOM files remain unchanged.",
        "2D views stream full-resolution slices on demand.",
        "3D volume uses optimized downsampled representation.",
    ],
}

with open(DATA_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(viewer_metadata, f, indent=2)

# -----------------------------
# Global context for next cells
# -----------------------------
SINUS_VIEWER_RUNTIME = {
    "runtime_dir": str(RUNTIME_DIR),
    "data_dir": str(DATA_DIR),
    "static_dir": str(STATIC_DIR),
    "metadata_path": str(DATA_DIR / "metadata.json"),
    "full_volume_path": str(DATA_DIR / "volume_full_i16.npy"),
    "volume3d_path": str(DATA_DIR / "volume3d_u8.npy"),
    "shape_zyx": volume_i16.shape,
    "volume3d_shape_zyx": vol_small_u8.shape,
}

# -----------------------------
# Report
# -----------------------------
full_mem_gib = volume_i16.nbytes / (1024**3)
small_mem_gib = vol_small_u8.nbytes / (1024**3)

print("[6/7] Runtime summary")
print("-" * 72)
print("Full volume shape (ZYX)     :", volume_i16.shape)
print("Full volume dtype           :", volume_i16.dtype)
print("Full volume RAM size        :", round(full_mem_gib, 3), "GiB")
print("3D optimized shape (ZYX)    :", vol_small_u8.shape)
print("3D optimized dtype          :", vol_small_u8.dtype)
print("3D optimized RAM size       :", round(small_mem_gib, 3), "GiB")
print("Runtime folder              :", RUNTIME_DIR)

print("\n[7/7] Dataset safety check")
print("-" * 72)
print("Original DICOM files modified : NO")
print("Original DICOM files moved    : NO")
print("Original DICOM files renamed  : NO")
print("Original DICOM files deleted  : NO")
print("Temporary runtime generated   : YES")

print("\n" + "=" * 72)
print("CELL 3 COMPLETE — VIEWER RUNTIME READY")
print("=" * 72)


# ===================== FROM VERIFIED SOURCE CELL 4 =====================

# ================================================================
# SINUS 3D VIEWER PRO — CELL 4
# FASTAPI + 3 MPR PLANES + INTERACTIVE VTK.JS VOLUME
# TEMPORARY RUNTIME — ORIGINAL DICOM FILES READ-ONLY
# ================================================================

import io, json, os, secrets, threading, time, socket, hmac
from pathlib import Path

import numpy as np
from PIL import Image
from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import (
    HTMLResponse, JSONResponse, Response, RedirectResponse
)
from starlette.middleware.base import BaseHTTPMiddleware
import uvicorn

if "SINUS_VIEWER_RUNTIME" not in globals():
    raise RuntimeError("Pehle Cell 3 run karo.")

DATA_PATH = Path(SINUS_VIEWER_RUNTIME["data_dir"])
with open(DATA_PATH / "metadata.json") as f:
    META = json.load(f)

# Memory-map full resolution CT for efficient slice access.
FULL = np.load(
    DATA_PATH / "volume_full_i16.npy",
    mmap_mode="r"
)

SMALL_PATH = DATA_PATH / "volume3d_u8.npy"
Z, Y, X = FULL.shape

# Random per-session credentials.
VIEWER_PASSWORD = secrets.token_urlsafe(18)
VIEWER_SESSION = secrets.token_urlsafe(32)
VIEWER_PORT = 8765

APP = FastAPI(
    title="Sinus 3D Viewer Pro",
    docs_url=None,
    redoc_url=None,
    openapi_url=None
)

def is_authorized(request):
    cookie = request.cookies.get("sinus_session", "")
    return hmac.compare_digest(cookie, VIEWER_SESSION)

class AuthMiddleware(BaseHTTPMiddleware):
    async def dispatch(self, request, call_next):
        allowed = {"/login", "/health"}
        if request.url.path in allowed:
            return await call_next(request)
        if not is_authorized(request):
            if request.url.path.startswith("/api/"):
                return JSONResponse(
                    {"error": "Authentication required"},
                    status_code=401
                )
            return RedirectResponse("/login")
        return await call_next(request)

APP.add_middleware(AuthMiddleware)

LOGIN_HTML = """
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sinus Viewer Login</title>
<style>
*{box-sizing:border-box}
body{margin:0;min-height:100vh;display:grid;place-items:center;
background:#090f19;color:#e8f1ff;font-family:Inter,Segoe UI,Arial}
.panel{width:min(410px,92vw);padding:36px;background:#131e2e;
border:1px solid #293a50;border-radius:18px}
h1{font-size:24px;margin:0 0 12px}
p{color:#96a8bf;font-size:14px;line-height:1.6}
input,button{width:100%;padding:14px;border-radius:9px;font-size:15px}
input{background:#0b1421;border:1px solid #40516a;color:white}
button{margin-top:14px;border:0;background:#328cff;color:white;
font-weight:700;cursor:pointer}
</style>
</head>
<body><div class="panel">
<h1>Sinus 3D Viewer Pro</h1>
<p>Private temporary imaging session. Enter the password
printed by Cell 4 in Colab.</p>
<form method="post" action="/login">
<input name="password" type="password" autocomplete="off"
placeholder="Session password" required autofocus>
<button type="submit">Open Viewer</button>
</form>
</div></body>
</html>
"""

from fastapi import Form

@APP.get("/login", response_class=HTMLResponse)
def login_page():
    return LOGIN_HTML

@APP.post("/login")
def login_submit(password: str = Form(...)):
    if not hmac.compare_digest(password, VIEWER_PASSWORD):
        return HTMLResponse(
            LOGIN_HTML.replace(
                "Private temporary imaging session.",
                "Incorrect password. Try again."
            ),
            status_code=401
        )
    resp = RedirectResponse("/", status_code=303)
    resp.set_cookie(
        "sinus_session", VIEWER_SESSION,
        httponly=True, samesite="strict",
        max_age=3600
    )
    return resp

@APP.get("/health")
def health():
    return {"status": "running"}

@APP.get("/api/meta")
def metadata():
    # No identifying patient name or source folder exposed.
    return {
        "shape": [Z, Y, X],
        "spacing": META["spacing_xyz_mm"],
        "direction": META["direction"],
        "origin": META["origin_xyz"],
        "fileCount": META["file_count"],
        "volumeShape": META["volume3d_shape_zyx"],
        "volumeSpacing": META["volume3d_spacing_xyz_mm"]
    }

def make_window(image, level, width):
    low = level - width / 2
    normalized = (image.astype(np.float32) - low) / width
    return np.uint8(np.clip(normalized, 0, 1) * 255)

@APP.get("/api/slice/{plane}/{index}")
def fetch_slice(
    plane: str,
    index: int,
    level: float = 400,
    width: float = 2000
):
    if not (-1100 <= level <= 3200 and 1 <= width <= 6000):
        raise HTTPException(400, "Invalid window parameters")

    if plane == "axial":
        if not 0 <= index < Z:
            raise HTTPException(400, "Invalid slice")
        data = FULL[index, :, :]
    elif plane == "coronal":
        if not 0 <= index < Y:
            raise HTTPException(400, "Invalid slice")
        data = FULL[:, index, :]
    elif plane == "sagittal":
        if not 0 <= index < X:
            raise HTTPException(400, "Invalid slice")
        data = FULL[:, :, index]
    else:
        raise HTTPException(404, "Unknown plane")

    # Flip vertical voxel axis for display.
    bitmap = Image.fromarray(
        make_window(np.flipud(data), level, width),
        mode="L"
    )
    output = io.BytesIO()
    bitmap.save(output, format="PNG", compress_level=1)

    return Response(
        output.getvalue(),
        media_type="image/png",
        headers={"Cache-Control": "no-store"}
    )

@APP.get("/api/volume")
def fetch_volume():
    return Response(
        SMALL_PATH.read_bytes()[128:] if False else
        np.load(SMALL_PATH).tobytes(order="C"),
        media_type="application/octet-stream",
        headers={"Cache-Control": "no-store"}
    )

VIEWER_HTML = r"""
<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sinus 3D Viewer Pro</title>
<script src="https://unpkg.com/vtk.js/dist/vtk.js"></script>
<style>
:root{--bg:#080e17;--panel:#101a29;--line:#26364b;
--muted:#8ba0b9;--text:#e8f1ff;--blue:#389bff}
*{box-sizing:border-box}
html,body{height:100%;margin:0;background:var(--bg);
font-family:Inter,Segoe UI,Arial,sans-serif;color:var(--text)}
body{display:flex;flex-direction:column;overflow:hidden}
header{height:62px;display:flex;align-items:center;justify-content:space-between;
padding:0 19px;background:#101927;border-bottom:1px solid var(--line);gap:12px}
.brand{font-size:17px;font-weight:750;letter-spacing:.2px}
.brand span{color:#42a8ff}
.sub{font-size:11px;color:var(--muted);margin-top:3px}
button,select,input[type=range]{cursor:pointer}
button,select{background:#1a2a3d;color:var(--text);
border:1px solid #30435b;border-radius:7px;padding:8px 10px}
button:hover{border-color:#4b9de8}
.toolbar{display:flex;align-items:center;gap:8px;flex-wrap:wrap}
main{flex:1;min-height:0;display:grid;grid-template-columns:1fr 1fr;
grid-template-rows:1fr 1fr;gap:8px;padding:8px}
.panel{min-width:0;min-height:0;position:relative;overflow:hidden;
border-radius:11px;background:#02060c;border:1px solid var(--line)}
.topbar{position:absolute;top:0;left:0;right:0;z-index:4;display:flex;
align-items:center;justify-content:space-between;padding:9px 12px;
background:linear-gradient(#091421ed,#09142177)}
.tag{font-size:11px;font-weight:750;letter-spacing:1.5px}
.count{font-size:11px;color:#a9bfda;font-variant-numeric:tabular-nums}
.viewport{position:absolute;inset:0;display:flex;align-items:center;
justify-content:center;touch-action:none;overflow:hidden}
.viewport canvas{display:block;max-width:none;image-rendering:auto}
.vtkview{position:absolute;inset:0;overflow:hidden}
.controls{position:absolute;bottom:9px;left:12px;right:12px;z-index:5;
display:flex;align-items:center;gap:10px;
background:#091522db;padding:8px 12px;border-radius:8px}
.controls input{flex:1;min-width:20px;accent-color:var(--blue)}
.status{font-size:11px;color:var(--muted)}
.foot{height:30px;display:flex;justify-content:space-between;
align-items:center;padding:0 16px;color:#92a5bb;font-size:11px;
border-top:1px solid var(--line)}
.full{position:fixed!important;inset:8px!important;z-index:99!important}
.panel:fullscreen{width:100vw;height:100vh;border-radius:0}
@media(max-width:780px){
header{height:auto;min-height:85px;align-items:flex-start;padding:10px}
main{grid-template-columns:1fr;grid-template-rows:repeat(4,minmax(300px,1fr));
overflow:auto}.toolbar{justify-content:flex-end}
}
</style>
</head>
<body>
<header>
<div><div class="brand"><span>◈</span> SINUS 3D VIEWER <span>PRO</span></div>
<div class="sub" id="scanInfo">Loading scan geometry...</div></div>
<div class="toolbar">
<select id="preset">
<option value="400,2000">Bone / Sinus</option>
<option value="40,400">Soft Tissue</option>
<option value="-600,1500">Air / Lung</option>
<option value="300,1500">Sinus Detail</option>
</select>
<button id="reset">Reset 2D</button>
<button id="reset3d">Reset 3D</button>
</div>
</header>
<main>
<section class="panel" id="p-axial">
<div class="topbar"><span class="tag" style="color:#f68b90">AXIAL</span>
<div><span class="count" id="n-axial"></span>
<button onclick="fullscreen('p-axial')">⛶</button></div></div>
<div class="viewport" id="v-axial"><canvas id="c-axial"></canvas></div>
<div class="controls"><input type="range" id="s-axial"><span class="status">Wheel: slices</span></div>
</section>

<section class="panel" id="p-coronal">
<div class="topbar"><span class="tag" style="color:#69dba0">CORONAL</span>
<div><span class="count" id="n-coronal"></span>
<button onclick="fullscreen('p-coronal')">⛶</button></div></div>
<div class="viewport" id="v-coronal"><canvas id="c-coronal"></canvas></div>
<div class="controls"><input type="range" id="s-coronal"><span class="status">Wheel: slices</span></div>
</section>

<section class="panel" id="p-sagittal">
<div class="topbar"><span class="tag" style="color:#efd16d">SAGITTAL</span>
<div><span class="count" id="n-sagittal"></span>
<button onclick="fullscreen('p-sagittal')">⛶</button></div></div>
<div class="viewport" id="v-sagittal"><canvas id="c-sagittal"></canvas></div>
<div class="controls"><input type="range" id="s-sagittal"><span class="status">Wheel: slices</span></div>
</section>

<section class="panel" id="p-3d">
<div class="topbar"><span class="tag" style="color:#77b5ff">3D VOLUME</span>
<div><span class="count" id="n-3d">Initializing...</span>
<button onclick="fullscreen('p-3d')">⛶</button></div></div>
<div id="view3d" class="vtkview"></div>
<div class="controls">
<span class="status">Bone opacity</span>
<input id="opacity" type="range" min="1" max="100" value="48">
<span class="status">Drag: rotate · Wheel: zoom</span>
</div>
</section>
</main>
<div class="foot"><span>Browser GPU · High-resolution 2D · Interactive 3D</span>
<span id="serverStatus">Connecting...</span></div>

<script>
const planes=["axial","coronal","sagittal"];
const images={}, state={}, controllers={};
let metadata, level=400,width=2000, vtkRenderer=null;
let vtkWindow=null, vtkOpacity=null;

function fullscreen(id){
 const el=document.getElementById(id);
 if(document.fullscreenElement) document.exitFullscreen();
 else el.requestFullscreen();
}
function clamp(x,a,b){return Math.max(a,Math.min(b,x));}
function status(s){document.getElementById("serverStatus").textContent=s;}

function draw(plane){
 const item=state[plane];
 if(!item || !item.img)return;
 const container=document.getElementById("v-"+plane);
 const canvas=document.getElementById("c-"+plane);
 const ctx=canvas.getContext("2d");
 const w=container.clientWidth,h=container.clientHeight;
 if(!w||!h)return;
 const dpr=Math.min(window.devicePixelRatio||1,2);
 canvas.width=Math.round(w*dpr);
 canvas.height=Math.round(h*dpr);
 canvas.style.width=w+"px";canvas.style.height=h+"px";
 ctx.setTransform(dpr,0,0,dpr,0,0);
 ctx.fillStyle="#02060c";ctx.fillRect(0,0,w,h);
 const iw=item.img.naturalWidth,ih=item.img.naturalHeight;
 // Physical aspect ratio, accounting for anisotropic voxels.
 const sx=metadata.spacing[plane==="sagittal"?1:0];
 const sy=metadata.spacing[plane==="axial"?1:2];
 const ar=(iw*sx)/(ih*sy);
 const fit=Math.min(w/ar,h)*.92;
 const dh=fit*item.zoom, dw=dh*ar;
 ctx.imageSmoothingEnabled=true;
 ctx.imageSmoothingQuality="high";
 ctx.drawImage(item.img,
   (w-dw)/2+item.dx,(h-dh)/2+item.dy,dw,dh);
}

async function loadSlice(plane){
 const item=state[plane];
 const seq=++item.seq;
 const url=`/api/slice/${plane}/${item.index}?level=${level}&width=${width}`;
 const img=new Image();
 img.onload=()=>{
  if(seq!==item.seq)return;
  item.img=img;draw(plane);
  document.getElementById("n-"+plane).textContent=
    `${item.index+1} / ${item.total}`;
 };
 img.onerror=()=>status("Slice loading error");
 img.src=url;
}

function initializePlanes(){
 const limits={axial:metadata.shape[0],
 coronal:metadata.shape[1],sagittal:metadata.shape[2]};
 planes.forEach(plane=>{
  const total=limits[plane];
  state[plane]={index:Math.floor(total/2),total,seq:0,
                  zoom:1,dx:0,dy:0,img:null};
  const slider=document.getElementById("s-"+plane);
  slider.min=0;slider.max=total-1;
  slider.value=state[plane].index;
  slider.oninput=()=>{
   state[plane].index=Number(slider.value);
   loadSlice(plane);
  };
  const surface=document.getElementById("v-"+plane);
  surface.addEventListener("wheel",e=>{
   e.preventDefault();
   const s=state[plane];
   if(e.ctrlKey){
    s.zoom=clamp(s.zoom*(e.deltaY<0?1.12:.89),.3,14);
    draw(plane);return;
   }
   s.index=clamp(s.index+(e.deltaY>0?1:-1),0,total-1);
   slider.value=s.index;
   loadSlice(plane);
  },{passive:false});
  let drag=null;
  surface.addEventListener("pointerdown",e=>{
   drag=[e.clientX,e.clientY];
   surface.setPointerCapture(e.pointerId);
  });
  surface.addEventListener("pointermove",e=>{
   if(!drag)return;
   const s=state[plane];
   s.dx+=e.clientX-drag[0];s.dy+=e.clientY-drag[1];
   drag=[e.clientX,e.clientY];draw(plane);
  });
  surface.addEventListener("pointerup",()=>drag=null);
  surface.addEventListener("pointercancel",()=>drag=null);
  surface.addEventListener("dblclick",()=>{
   const s=state[plane];s.zoom=1;s.dx=0;s.dy=0;draw(plane);
  });
  loadSlice(plane);
 });
}

async function initialize3D(){
 const label=document.getElementById("n-3d");
 try{
 if(!window.vtk)throw Error("vtk.js library not loaded");
 const shape=metadata.volumeShape;
 const spacing=metadata.volumeSpacing;
 const resp=await fetch("/api/volume");
 if(!resp.ok)throw Error("Volume download failed");
 const raw=new Uint8Array(await resp.arrayBuffer());
 if(raw.length!==shape[0]*shape[1]*shape[2])
   throw Error("3D volume dimensions mismatch");

 const V=vtk;
 const data=V.Common.DataModel.vtkImageData.newInstance();
 data.setDimensions(shape[2],shape[1],shape[0]);
 data.setSpacing(...spacing);
 // Spatial direction is retained; 2D medical orientation will
 // be separately validated before diagnostic interpretation.
 if(typeof data.setDirection==="function"){
   data.setDirection(metadata.direction);
 }
 const scalars=V.Common.Core.vtkDataArray.newInstance({
   name:"CTIntensity",values:raw,numberOfComponents:1
 });
 data.getPointData().setScalars(scalars);

 const mapper=V.Rendering.Core.vtkVolumeMapper.newInstance();
 mapper.setInputData(data);
 mapper.setSampleDistance(Math.min(...spacing)*1.3);

 const vol=V.Rendering.Core.vtkVolume.newInstance();
 vol.setMapper(mapper);

 const color=V.Rendering.Core.vtkColorTransferFunction.newInstance();
 // Values from Cell 3 are mapped to 0..255.
 color.addRGBPoint(0,0,0,0);
 color.addRGBPoint(70,.18,.24,.31);
 color.addRGBPoint(120,.65,.68,.72);
 color.addRGBPoint(190,.93,.89,.82);
 color.addRGBPoint(255,1,1,1);

 const opacity=V.Common.DataModel.vtkPiecewiseFunction.newInstance();
 vtkOpacity=opacity;

 function applyOpacity(){
   const strength=Number(document.getElementById("opacity").value)/100;
   opacity.removeAllPoints();
   opacity.addPoint(0,0);
   opacity.addPoint(105,0);
   opacity.addPoint(115,.002*strength);
   opacity.addPoint(140,.035*strength);
   opacity.addPoint(180,.19*strength);
   opacity.addPoint(255,.50*strength);
   if(vtkWindow)vtkWindow.render();
 }

 vol.getProperty().setRGBTransferFunction(0,color);
 vol.getProperty().setScalarOpacity(0,opacity);
 vol.getProperty().setInterpolationTypeToLinear();
 vol.getProperty().setShade(true);
 vol.getProperty().setAmbient(.25);
 vol.getProperty().setDiffuse(.75);
 vol.getProperty().setSpecular(.15);
 vol.getProperty().setScalarOpacityUnitDistance(0,Math.min(...spacing)*2);

 const fullscreenRenderer=
   V.Rendering.Misc.vtkFullScreenRenderWindow.newInstance({
    rootContainer:document.getElementById("view3d"),
    containerStyle:{height:"100%",width:"100%",position:"absolute"},
    background:[.008,.016,.029]
   });
 vtkRenderer=fullscreenRenderer.getRenderer();
 vtkWindow=fullscreenRenderer.getRenderWindow();
 vtkRenderer.addVolume(vol);
 vtkRenderer.resetCamera();
 applyOpacity();
 vtkWindow.render();
 document.getElementById("opacity").oninput=applyOpacity;
 label.textContent=shape.join(" × ")+" voxels";
 document.getElementById("reset3d").onclick=()=>{
   vtkRenderer.resetCamera();vtkWindow.render();
 };
 }catch(e){
  console.error(e);
  label.textContent="3D error: "+e.message;
  status("3D setup failed — see browser console");
 }
}

document.getElementById("preset").onchange=e=>{
 [level,width]=e.target.value.split(",").map(Number);
 planes.forEach(loadSlice);
};
document.getElementById("reset").onclick=()=>{
 planes.forEach(p=>{
  Object.assign(state[p],{zoom:1,dx:0,dy:0});
  draw(p);
 });
};

window.addEventListener("resize",()=>{
 planes.forEach(draw);
 if(vtkWindow)vtkWindow.render();
});
document.addEventListener("fullscreenchange",()=>{
 setTimeout(()=>{
  planes.forEach(draw);
  if(vtkWindow)vtkWindow.render();
 },100);
});

async function main(){
 try{
 const r=await fetch("/api/meta");
 if(!r.ok)throw Error("Metadata API failed");
 metadata=await r.json();
 document.getElementById("scanInfo").textContent=
   `${metadata.shape.join(" × ")} voxels · ${metadata.fileCount} DICOM slices`;
 initializePlanes();
 status("2D connected · Preparing 3D");
 await initialize3D();
 if(vtkWindow)status("Viewer online");
 }catch(e){
  status("Connection error: "+e.message);console.error(e);
 }
}
main();
</script>
</body>
</html>
"""

@APP.get("/", response_class=HTMLResponse)
def homepage():
    return HTMLResponse(
        VIEWER_HTML,
        headers={"Cache-Control": "no-store"}
    )

# Start or reuse backend.
if "SINUS_SERVER" in globals() and SINUS_SERVER.started:
    SINUS_SERVER.should_exit = True
    time.sleep(2)

config = uvicorn.Config(
    APP, host="127.0.0.1", port=VIEWER_PORT,
    log_level="warning", access_log=False
)
SINUS_SERVER = uvicorn.Server(config)
SINUS_SERVER_THREAD = threading.Thread(
    target=SINUS_SERVER.run, daemon=True
)
SINUS_SERVER_THREAD.start()

for _ in range(40):
    if SINUS_SERVER.started:
        break
    time.sleep(.25)

if not SINUS_SERVER.started:
    raise RuntimeError("Backend failed to start.")

import requests
response = requests.get(
    f"http://127.0.0.1:{VIEWER_PORT}/health", timeout=5
)
response.raise_for_status()

print("=" * 65)
print("SINUS 3D VIEWER PRO — CELL 4")
print("=" * 65)
print("Backend status : ONLINE")
print("Local address  : http://127.0.0.1:8765")
print("2D viewports   : Axial, Coronal, Sagittal")
print("3D viewport    : vtk.js WebGL Volume")
print("Original DICOM : UNCHANGED")
print()
print("PRIVATE SESSION PASSWORD:")
print(VIEWER_PASSWORD)
print()
print("Password kisi public chat ya screenshot mein share na karein.")
print("External HTTPS link will be configured separately.")
print("=" * 65)
print("CELL 4 COMPLETE")
print("=" * 65)



# ===================== FROM VERIFIED SOURCE CELL 5 =====================

# ============================================================
# SINUS 3D VIEWER PRO — CELL 4B
# 3D ENGINE REPAIR + IMAGE CLARITY + DIAGNOSTICS
# EXISTING BACKEND / TUNNEL REUSED
# ============================================================

import requests
import secrets
import re
import numpy as np
from fastapi.responses import Response

print("=" * 68)
print("SINUS VIEWER PRO — 3D ENGINE REPAIR")
print("=" * 68)

if "APP" not in globals() or "VIEWER_HTML" not in globals():
    raise RuntimeError("Cell 4 backend context missing.")

if "SINUS_SERVER" not in globals() or not SINUS_SERVER.started:
    raise RuntimeError("Cell 4 server is not running.")

# ------------------------------------------------------------
# 1. Fetch known vtk.js browser bundle into Colab RAM
# ------------------------------------------------------------

print("\n[1/6] Loading vtk.js browser rendering engine...")

vtk_urls = [
    "https://cdn.jsdelivr.net/npm/vtk.js@35.7.1/vtk.js",
    "https://unpkg.com/vtk.js@35.7.1/vtk.js"
]

VTK_BUNDLE = None

for url in vtk_urls:
    try:
        r = requests.get(url, timeout=45)
        r.raise_for_status()

        if len(r.content) < 500_000:
            raise ValueError("Unexpectedly small JavaScript bundle")

        VTK_BUNDLE = r.content

        print("vtk.js downloaded successfully.")
        print("Bundle size:", round(len(VTK_BUNDLE) / 1048576, 2), "MiB")
        break

    except Exception as e:
        print("Source unavailable:", type(e).__name__)

if VTK_BUNDLE is None:
    raise RuntimeError(
        "Could not download vtk.js. Existing viewer unchanged."
    )

# Serve JS from existing authenticated backend.
# Add route only once; updated global bytes work on reruns.
if not globals().get("VTK_ROUTE_INSTALLED", False):
    @APP.get("/vendor/vtk.js")
    def serve_vtk_bundle():
        return Response(
            VTK_BUNDLE,
            media_type="application/javascript",
            headers={"Cache-Control": "no-store"}
        )

    VTK_ROUTE_INSTALLED = True

# ------------------------------------------------------------
# 2. Replace external JS library dependency
# ------------------------------------------------------------

print("[2/6] Replacing browser rendering engine...")

VIEWER_HTML = re.sub(
    r'<script src="https://unpkg\.com/vtk\.js/dist/vtk\.js"></script>',
    '<script src="/vendor/vtk.js"></script>',
    VIEWER_HTML
)

# Keep future execution idempotent.
VIEWER_HTML = VIEWER_HTML.replace(
    '<script src="https://cdn.jsdelivr.net/npm/vtk.js@35.7.1/vtk.js"></script>',
    '<script src="/vendor/vtk.js"></script>'
)

if '<script src="/vendor/vtk.js"></script>' not in VIEWER_HTML:
    raise RuntimeError("Could not locate vtk.js script tag.")

# ------------------------------------------------------------
# 3. Replace 3D initializer
# ------------------------------------------------------------

print("[3/6] Installing improved 3D volume renderer...")

NEW_3D_JS = r"""
let vtkFullScreen = null;
let vtkCameraRenderer = null;
let vtkRenderWindow = null;
let vtkVolumeProperty = null;

async function initialize3D() {
 const label = document.getElementById("n-3d");
 const panel = document.getElementById("view3d");

 function report(msg) {
   label.textContent = msg;
   console.log("[SINUS 3D]", msg);
 }

 try {
   report("Checking WebGL...");

   const testCanvas = document.createElement("canvas");
   const gl = testCanvas.getContext("webgl2", {
     failIfMajorPerformanceCaveat: false
   });

   if (!gl) {
     throw Error("WebGL2 unavailable. Enable browser hardware acceleration.");
   }

   if (!window.vtk) {
     throw Error("vtk.js failed to initialize.");
   }

   report("Downloading CT volume...");

   const response = await fetch("/api/volume", {
     cache: "no-store"
   });

   if (!response.ok) {
     throw Error("Volume API returned HTTP " + response.status);
   }

   const buffer = await response.arrayBuffer();
   const voxels = new Uint8Array(buffer);

   const dims = metadata.volumeShape;
   const spacing = metadata.volumeSpacing;
   const expected = dims[0] * dims[1] * dims[2];

   if (voxels.length !== expected) {
     throw Error(
       "Voxel mismatch: expected " + expected +
       ", got " + voxels.length
     );
   }

   report("Building 3D volume...");

   const V = window.vtk;

   // NumPy volume Z,Y,X -> vtkImageData X,Y,Z
   const image = V.Common.DataModel.vtkImageData.newInstance();

   image.setDimensions(dims[2], dims[1], dims[0]);
   image.setSpacing(spacing[0], spacing[1], spacing[2]);
   image.setOrigin(0, 0, 0);

   // Preserve direction cosines from original image metadata.
   if (typeof image.setDirection === "function") {
     image.setDirection(metadata.direction);
   }

   const scalars = V.Common.Core.vtkDataArray.newInstance({
     name: "CTVolume",
     values: voxels,
     numberOfComponents: 1
   });

   image.getPointData().setScalars(scalars);

   const mapper = V.Rendering.Core.vtkVolumeMapper.newInstance();
   mapper.setInputData(image);

   // Larger sample step improves interaction speed.
   mapper.setSampleDistance(
     Math.max(0.5, Math.min(...spacing) * 1.5)
   );

   const volume = V.Rendering.Core.vtkVolume.newInstance();
   volume.setMapper(mapper);

   // The uint8 volume represents roughly -1024..2000 HU.
   const colors =
     V.Rendering.Core.vtkColorTransferFunction.newInstance();

   colors.addRGBPoint(0,   0.01, 0.01, 0.02);
   colors.addRGBPoint(95,  0.20, 0.27, 0.34);
   colors.addRGBPoint(112, 0.58, 0.64, 0.71);
   colors.addRGBPoint(150, 0.87, 0.85, 0.78);
   colors.addRGBPoint(255, 1.00, 0.98, 0.91);

   const opacity =
     V.Common.DataModel.vtkPiecewiseFunction.newInstance();

   const prop = volume.getProperty();
   prop.setRGBTransferFunction(0, colors);
   prop.setScalarOpacity(0, opacity);
   prop.setInterpolationTypeToLinear();
   prop.setShade(false);
   prop.setScalarOpacityUnitDistance(
     0, Math.max(0.5, Math.min(...spacing) * 2)
   );

   vtkVolumeProperty = prop;

   // Clear any previously attached renderer.
   if (vtkFullScreen) {
     vtkFullScreen.delete();
     vtkFullScreen = null;
   }

   panel.innerHTML = "";

   vtkFullScreen =
     V.Rendering.Misc.vtkFullScreenRenderWindow.newInstance({
       rootContainer: panel,
       containerStyle: {
         position: "absolute",
         width: "100%",
         height: "100%",
         overflow: "hidden"
       },
       background: [0.008, 0.012, 0.025]
     });

   vtkCameraRenderer = vtkFullScreen.getRenderer();
   vtkRenderWindow = vtkFullScreen.getRenderWindow();

   vtkRenderer = vtkCameraRenderer;
   vtkWindow = vtkRenderWindow;

   vtkCameraRenderer.addVolume(volume);
   vtkCameraRenderer.resetCamera();

   function updateOpacity() {
     const value = Number(
       document.getElementById("opacity").value
     ) / 100;

     opacity.removeAllPoints();
     opacity.addPoint(0, 0);
     opacity.addPoint(100, 0);
     opacity.addPoint(111, 0);
     opacity.addPoint(120, 0.04 * value);
     opacity.addPoint(145, 0.23 * value);
     opacity.addPoint(180, 0.52 * value);
     opacity.addPoint(255, 0.78 * value);

     vtkRenderWindow.render();
   }

   document.getElementById("opacity").oninput = updateOpacity;

   document.getElementById("reset3d").onclick = () => {
     vtkCameraRenderer.resetCamera();
     vtkFullScreen.resize();
     vtkRenderWindow.render();
   };

   // Maintain proper WebGL canvas size.
   const observer = new ResizeObserver(() => {
     if (vtkFullScreen) {
       vtkFullScreen.resize();
       vtkRenderWindow.render();
     }
   });

   observer.observe(panel);

   updateOpacity();

   report("3D READY · " + dims.join(" × "));
   status("3D engine ready");

 } catch (err) {
   console.error("[SINUS 3D ERROR]", err);
   const message = String(err.message || err);
   label.textContent = "3D ERROR";

   panel.innerHTML =
     '<div style="padding:65px 20px;color:#ff9f9f;' +
     'font:14px Segoe UI,Arial;line-height:1.8">' +
     '<b>3D initialization failed</b>' +
     '<div id="vtkErrorMessage"></div>' +
     '<p>Check browser F12 → Console.</p></div>';

   document.getElementById("vtkErrorMessage").textContent = message;
   status("3D ERROR: " + message);
 }
}
"""

start_marker = "async function initialize3D(){"
end_marker = 'document.getElementById("preset").onchange='

start = VIEWER_HTML.find(start_marker)
end = VIEWER_HTML.find(end_marker, start)

if start == -1 or end == -1:
    # Support rerunning the patch.
    start_marker = "let vtkFullScreen = null;"
    start = VIEWER_HTML.find(start_marker)
    end = VIEWER_HTML.find(end_marker, start)

if start == -1 or end == -1:
    raise RuntimeError("Could not locate existing 3D initializer.")

VIEWER_HTML = (
    VIEWER_HTML[:start] +
    NEW_3D_JS + "\n" +
    VIEWER_HTML[end:]
)

# ------------------------------------------------------------
# 4. Improve 2D image display
# ------------------------------------------------------------

print("[4/6] Improving CT slice rendering...")

# Preserve native sharp pixels when zooming in.
VIEWER_HTML = VIEWER_HTML.replace(
    'ctx.imageSmoothingEnabled=true;',
    'ctx.imageSmoothingEnabled=(item.zoom<1.5);'
)

# Allow higher DPI canvas rendering.
VIEWER_HTML = VIEWER_HTML.replace(
    'Math.min(window.devicePixelRatio||1,2)',
    'Math.min(window.devicePixelRatio||1,3)'
)

# Add browser error monitoring.
monitor_js = r"""
window.addEventListener("error", event => {
 console.error("Viewer JavaScript error:", event.message);
 const s=document.getElementById("serverStatus");
 if(s) s.textContent="Browser error: "+event.message;
});
window.addEventListener("unhandledrejection", event => {
 console.error("Viewer Promise error:", event.reason);
});
"""

VIEWER_HTML = VIEWER_HTML.replace(
    '</script>\n</body>',
    monitor_js + '\n</script>\n</body>'
)

# ------------------------------------------------------------
# 5. Rotate credentials
# ------------------------------------------------------------

print("[5/6] Rotating previously exposed credentials...")

VIEWER_PASSWORD = secrets.token_urlsafe(24)
VIEWER_SESSION = secrets.token_urlsafe(32)

# Authentication middleware reads these globals at request time.
# Old login cookies will no longer be accepted.

# ------------------------------------------------------------
# 6. Validate running API
# ------------------------------------------------------------

print("[6/6] Validating backend and asset endpoints...")

base = "http://127.0.0.1:8765"

health = requests.get(base + "/health", timeout=10)
health.raise_for_status()

with requests.Session() as session:
    session.cookies.set("sinus_session", VIEWER_SESSION)

    page = session.get(base + "/", timeout=10)
    script = session.get(base + "/vendor/vtk.js", timeout=20)
    volume_response = session.get(
        base + "/api/volume", timeout=30
    )

    assert page.status_code == 200
    assert script.status_code == 200
    assert len(script.content) > 500_000

    expected_size = int(np.prod(
        META["volume3d_shape_zyx"]
    ))

    assert volume_response.status_code == 200
    assert len(volume_response.content) == expected_size

print("\n" + "=" * 68)
print("CELL 4B — PATCH INSTALLED")
print("=" * 68)
print("Backend status       : ONLINE")
print("vtk.js asset         : VERIFIED")
print("3D voxel data       : VERIFIED")
print("2D enhancement      : INSTALLED")
print("Private credentials : ROTATED")
print("Original DICOM      : UNCHANGED")
print()
print("New private password (DO NOT SHARE):")
print(VIEWER_PASSWORD)
print()
print("Existing external HTTPS tunnel can be reused.")
print("Browser rendering still needs testing.")
print("=" * 68)



# ===================== FROM VERIFIED SOURCE CELL 6 =====================

# ================================================================
# SINUS 3D VIEWER PRO — CELL 4C
# HIGH-QUALITY 3D VOLUME OPTIMIZATION
# SAFE: ORIGINAL DICOM FILES ARE READ-ONLY
# ================================================================

import os
import json
import time
import numpy as np
import SimpleITK as sitk
from pathlib import Path

print("=" * 70)
print("SINUS VIEWER PRO — 3D QUALITY OPTIMIZATION")
print("=" * 70)

required = [
    "CT_IMAGE", "CT_ARRAY", "META",
    "SINUS_VIEWER_RUNTIME", "SINUS_SERVER"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError(f"Missing previous cells: {missing}")

if not SINUS_SERVER.started:
    raise RuntimeError("Viewer backend is not running.")

print("\nAvailable 3D quality levels:")
print("  1 — FAST       (192 max dimension)")
print("  2 — BALANCED   (256 max dimension) [Recommended]")
print("  3 — ULTRA      (320 max dimension)")
print()
print("Higher quality uses more browser GPU memory.")

choice = "2"  # automatic Balanced 256 quality

levels = {"1": 192, "2": 256, "3": 320}

if choice not in levels:
    raise ValueError("Enter 1, 2, or 3.")

target = levels[choice]
start = time.time()

# ------------------------------------------------------------
# Validate source geometry
# ------------------------------------------------------------

original_size = CT_IMAGE.GetSize()
original_spacing = CT_IMAGE.GetSpacing()
direction = np.array(
    CT_IMAGE.GetDirection(), dtype=float
).reshape(3, 3)

extent_mm = [
    (n - 1) * s
    for n, s in zip(original_size, original_spacing)
]

print("\n[1/5] Physical CT geometry")
print("Original dimensions XYZ:", original_size)
print("Original spacing XYZ:", original_spacing)
print("Physical extent mm:", tuple(round(x, 2) for x in extent_mm))
print("Direction matrix:")
print(direction)

if min(extent_mm) < 80:
    print("\nWARNING: Unusually small physical scan extent.")
    print("Verify source DICOM PixelSpacing / slice geometry.")
    print("Do not assume physical measurements are correct.")

# ------------------------------------------------------------
# Create resampled volume using SimpleITK
# ------------------------------------------------------------

print("\n[2/5] Resampling CT volume...")

scale = min(1.0, target / max(original_size))

new_size = [
    max(2, round(n * scale))
    for n in original_size
]

# Preserve first and last voxel-center positions.
new_spacing = [
    s * (n - 1) / (m - 1)
    for n, m, s in zip(
        original_size, new_size, original_spacing
    )
]

resampler = sitk.ResampleImageFilter()

resampler.SetSize([int(v) for v in new_size])
resampler.SetOutputSpacing([float(v) for v in new_spacing])
resampler.SetOutputOrigin(CT_IMAGE.GetOrigin())
resampler.SetOutputDirection(CT_IMAGE.GetDirection())
resampler.SetTransform(sitk.Transform())
resampler.SetInterpolator(sitk.sitkLinear)
resampler.SetDefaultPixelValue(-1024)
resampler.SetOutputPixelType(sitk.sitkInt16)
resampler.SetNumberOfThreads(8)

reduced_image = resampler.Execute(CT_IMAGE)
reduced = sitk.GetArrayFromImage(reduced_image)

print("Optimized dimensions ZYX:", reduced.shape)

# ------------------------------------------------------------
# Create browser-ready intensity data
# ------------------------------------------------------------

print("\n[3/5] Encoding browser-friendly volume...")

# Maintain same intensity mapping as previous Cell 3.
normalized = np.clip(
    (reduced.astype(np.float32) + 1024.0)
    * (255.0 / 3024.0),
    0,
    255
).astype(np.uint8)

# ------------------------------------------------------------
# Atomic replacement of temporary runtime files
# ------------------------------------------------------------

print("\n[4/5] Updating temporary viewer volume...")

data_dir = Path(SINUS_VIEWER_RUNTIME["data_dir"])
volume_path = data_dir / "volume3d_u8.npy"
temporary_path = data_dir / "volume3d_pending.npy"

with open(temporary_path, "wb") as f:
    np.save(f, normalized)

# Replace only the temporary cached 3D file.
# Original patient DICOM files are never written.
os.replace(temporary_path, volume_path)

updated_meta = dict(META)
updated_meta["volume3d_shape_zyx"] = [
    int(v) for v in normalized.shape
]
updated_meta["volume3d_spacing_xyz_mm"] = [
    float(v) for v in new_spacing
]

meta_path = data_dir / "metadata.json"
pending_meta = data_dir / "metadata_pending.json"

with open(pending_meta, "w") as f:
    json.dump(updated_meta, f, indent=2)

os.replace(pending_meta, meta_path)

# Existing FastAPI endpoints read this global metadata.
META.update(updated_meta)

SINUS_VIEWER_RUNTIME["volume3d_shape_zyx"] = normalized.shape

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\n[5/5] Verifying updated volume...")

loaded = np.load(volume_path, mmap_mode="r")

assert loaded.shape == normalized.shape
assert loaded.dtype == np.uint8
assert tuple(META["volume3d_shape_zyx"]) == loaded.shape

print("\n" + "=" * 70)
print("3D QUALITY OPTIMIZATION COMPLETE")
print("=" * 70)

print("Quality level        :", choice)
print("3D volume shape ZYX  :", loaded.shape)
print("3D data size         :", round(loaded.nbytes / 1048576, 2), "MiB")
print("Spacing XYZ (mm)     :", tuple(round(x, 5) for x in new_spacing))
print("Processing seconds   :", round(time.time() - start, 2))
print("Backend              : PRESERVED")
print("External link        : PRESERVED")
print("Original DICOM files : UNCHANGED")
print()
print("Refresh browser to load new 3D volume.")
print("=" * 70)



# ===================== FROM VERIFIED SOURCE CELL 8 =====================

# ============================================================
# SINUS 3D VIEWER PRO — CELL 4E
# BROWSER-RESIDENT MPR — NO NETWORK PER SLICE
# PRESERVES EXISTING WORKING 3D ENGINE
# ORIGINAL PATIENT FILES READ-ONLY
# ============================================================

import os
import re
import time
import threading
import numpy as np
import requests
from pathlib import Path
from fastapi import HTTPException
from fastapi.responses import FileResponse

print("=" * 68)
print("SINUS VIEWER — BROWSER-RESIDENT MPR")
print("=" * 68)

needed = ["APP", "VIEWER_HTML", "FULL", "META", "SINUS_SERVER"]
if any(k not in globals() for k in needed):
    raise RuntimeError("Earlier viewer cells are not active.")
if not SINUS_SERVER.started:
    raise RuntimeError("Backend not running.")

# ----------------------------------------------
# 1. Create an 8-bit, native-resolution volume
# ----------------------------------------------
MPR_DIR = Path("/content/sinus_viewer_runtime/mpr")
MPR_DIR.mkdir(parents=True, exist_ok=True)

MPR_PRESETS = {
    (400, 2000): "bone",
    (40, 400): "soft",
    (-600, 1500): "air",
    (300, 1500): "sinus"
}
MPR_LOCK = threading.Lock()

def build_mpr_file(level, width):
    key = (int(level), int(width))
    if key not in MPR_PRESETS:
        raise ValueError("Unsupported preset")

    path = MPR_DIR / (MPR_PRESETS[key] + ".bin")
    if path.exists() and path.stat().st_size == FULL.size:
        return path

    with MPR_LOCK:
        if path.exists() and path.stat().st_size == FULL.size:
            return path

        tmp = path.with_suffix(".pending")
        low = level - width / 2
        scale = 255.0 / width

        out = np.memmap(
            str(tmp), dtype=np.uint8,
            mode="w+", shape=FULL.shape
        )

        # Chunked processing to avoid large temporary arrays.
        for z in range(FULL.shape[0]):
            frame = FULL[z].astype(np.float32)
            frame = np.clip((frame - low) * scale, 0, 255)
            out[z] = frame.astype(np.uint8)

            if z % 128 == 0:
                print(f"  Encoding: {z}/{FULL.shape[0]}")

        out.flush()
        del out
        os.replace(tmp, path)
        return path

print("\n[1/4] Preparing native-resolution bone volume...")
start_time = time.time()
bone_path = build_mpr_file(400, 2000)
print("Prepared:", round(bone_path.stat().st_size/1048576, 1), "MiB")

# ----------------------------------------------
# 2. Protected volume endpoint
# ----------------------------------------------
if not globals().get("SINUS_MPR_ROUTE_READY", False):
    @APP.get("/api/mpr-volume")
    def serve_mpr_volume(level: int = 400, width: int = 2000):
        if (level, width) not in MPR_PRESETS:
            raise HTTPException(400, "Unsupported CT preset")

        p = build_mpr_file(level, width)
        return FileResponse(
            p,
            media_type="application/octet-stream",
            headers={
                "Cache-Control": "no-store",
                "X-Content-Type-Options": "nosniff"
            }
        )

    SINUS_MPR_ROUTE_READY = True

# ----------------------------------------------
# 3. Replace old 2D JS with local-memory engine
# ----------------------------------------------
NEW_MPR_JS = r"""
// ==========================================================
// SINUS BROWSER-RESIDENT MPR ENGINE
// ==========================================================
const MPR = {
  voxels:null,
  loading:false,
  ready:false,
  generation:0,
  raf:{},
  scratch:{}
};

function mprMessage(msg){
  status(msg);
  const el=document.getElementById("scanInfo");
  if(el) el.textContent=msg;
}

async function reloadMPRVolume(){
  const generation=++MPR.generation;
  MPR.ready=false;
  MPR.loading=true;
  MPR.voxels=null;

  const expected=metadata.shape.reduce((a,b)=>a*b,1);
  const url=`/api/mpr-volume?level=${level}&width=${width}`;

  mprMessage("Loading local-memory CT volume...");
  try{
    const res=await fetch(url,{cache:"no-store"});
    if(!res.ok) throw Error("HTTP "+res.status);

    // Fetch once. No server request is needed during scrolling.
    const buffer=await res.arrayBuffer();
    if(generation!==MPR.generation)return;

    if(buffer.byteLength!==expected){
      throw Error(
        "Volume size mismatch: "+buffer.byteLength+
        " vs "+expected
      );
    }

    MPR.voxels=new Uint8Array(buffer);
    MPR.ready=true;
    MPR.loading=false;

    for(const p of planes){
      scheduleMPRDraw(p);
    }
    mprMessage(
      "LOCAL MPR READY · "+
      (buffer.byteLength/1048576).toFixed(1)+
      " MiB · No per-slice network requests"
    );
  }catch(e){
    MPR.loading=false;
    console.error("[LOCAL MPR]",e);
    mprMessage("MPR ERROR: "+e.message);
  }
}

function scheduleMPRDraw(plane){
  if(MPR.raf[plane])return;
  MPR.raf[plane]=requestAnimationFrame(()=>{
    MPR.raf[plane]=null;
    draw(plane);
  });
}

function draw(plane){
  if(!MPR.ready || !MPR.voxels)return;
  const s=state[plane];
  if(!s)return;

  const [Z,Y,X]=metadata.shape;
  const sp=metadata.spacing;
  const volume=MPR.voxels;

  let iw,ih,sx,sy;

  if(plane==="axial"){
    iw=X;ih=Y;sx=sp[0];sy=sp[1];
  }else if(plane==="coronal"){
    iw=X;ih=Z;sx=sp[0];sy=sp[2];
  }else{
    iw=Y;ih=Z;sx=sp[1];sy=sp[2];
  }

  // Reuse scratch buffers. Avoid per-frame allocations.
  let scratch=MPR.scratch[plane];
  if(!scratch || scratch.w!==iw || scratch.h!==ih){
    const off=document.createElement("canvas");
    off.width=iw;off.height=ih;
    const offctx=off.getContext("2d",{alpha:false});
    scratch={
      canvas:off,ctx:offctx,w:iw,h:ih,
      pixels:offctx.createImageData(iw,ih)
    };
    MPR.scratch[plane]=scratch;
  }

  const rgba=scratch.pixels.data;
  const index=s.index;

  // Native-resolution slice extraction.
  // Note: physical patient orientation must be verified separately.
  if(plane==="axial"){
    const base=index*Y*X;
    for(let row=0;row<Y;row++){
      const src=base+(Y-1-row)*X;
      let dst=row*X*4;
      for(let col=0;col<X;col++){
        const v=volume[src+col];
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=255;
      }
    }
  }else if(plane==="coronal"){
    for(let row=0;row<Z;row++){
      const z=Z-1-row;
      const base=(z*Y+index)*X;
      let dst=row*X*4;
      for(let col=0;col<X;col++){
        const v=volume[base+col];
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=255;
      }
    }
  }else{
    for(let row=0;row<Z;row++){
      const z=Z-1-row;
      let dst=row*Y*4;
      for(let col=0;col<Y;col++){
        const v=volume[(z*Y+col)*X+index];
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=v;
        rgba[dst++]=255;
      }
    }
  }

  scratch.ctx.putImageData(scratch.pixels,0,0);

  const container=document.getElementById("v-"+plane);
  const canvas=document.getElementById("c-"+plane);
  const ctx=canvas.getContext("2d",{alpha:false});

  const w=container.clientWidth;
  const h=container.clientHeight;
  if(!w || !h)return;

  const dpr=Math.min(window.devicePixelRatio||1,2);
  const cw=Math.round(w*dpr);
  const ch=Math.round(h*dpr);

  if(canvas.width!==cw || canvas.height!==ch){
    canvas.width=cw;
    canvas.height=ch;
  }

  canvas.style.width=w+"px";
  canvas.style.height=h+"px";
  ctx.setTransform(dpr,0,0,dpr,0,0);
  ctx.fillStyle="#02060c";
  ctx.fillRect(0,0,w,h);

  const physicalW=iw*sx;
  const physicalH=ih*sy;
  const fit=Math.min(w/physicalW,h/physicalH)*0.97;
  const dw=physicalW*fit*s.zoom;
  const dh=physicalH*fit*s.zoom;

  // High-quality downscaling; native pixels on zoom-in.
  ctx.imageSmoothingEnabled=(dw/iw<1);
  ctx.imageSmoothingQuality="high";

  ctx.drawImage(
    scratch.canvas,
    (w-dw)/2+s.dx,
    (h-dh)/2+s.dy,
    dw,dh
  );

  document.getElementById("n-"+plane).textContent=
    `${s.index+1} / ${s.total}`;
}

function loadSlice(plane){
  scheduleMPRDraw(plane);
}

function initializePlanes(){
  const limits={
    axial:metadata.shape[0],
    coronal:metadata.shape[1],
    sagittal:metadata.shape[2]
  };

  planes.forEach(plane=>{
    const total=limits[plane];
    const s=state[plane]={
      index:Math.floor(total/2),
      total,zoom:1,dx:0,dy:0,wheel:0
    };

    const slider=document.getElementById("s-"+plane);
    const surface=document.getElementById("v-"+plane);
    slider.min=0;slider.max=total-1;
    slider.value=s.index;

    const moveTo=i=>{
      const n=clamp(Math.round(i),0,total-1);
      if(n===s.index)return;
      s.index=n;
      slider.value=n;
      scheduleMPRDraw(plane);
    };

    slider.oninput=()=>moveTo(Number(slider.value));

    surface.addEventListener("wheel",e=>{
      e.preventDefault();

      let delta=e.deltaY;
      if(e.deltaMode===1)delta*=16;
      if(e.deltaMode===2)delta*=surface.clientHeight;

      if(e.ctrlKey){
        s.zoom=clamp(
          s.zoom*Math.exp(-delta*.0015),
          .25,16
        );
        scheduleMPRDraw(plane);
        return;
      }

      s.wheel+=delta;
      // Typical mouse notch ~100; touchpad fractional input
      // accumulates before advancing a slice.
      const steps=Math.trunc(s.wheel/70);
      if(steps){
        s.wheel-=steps*70;
        moveTo(s.index+steps);
      }
    },{passive:false});

    let drag=null;
    surface.addEventListener("pointerdown",e=>{
      drag=[e.clientX,e.clientY];
      surface.setPointerCapture(e.pointerId);
    });
    surface.addEventListener("pointermove",e=>{
      if(!drag)return;
      s.dx+=e.clientX-drag[0];
      s.dy+=e.clientY-drag[1];
      drag=[e.clientX,e.clientY];
      scheduleMPRDraw(plane);
    });
    const stop=()=>drag=null;
    surface.addEventListener("pointerup",stop);
    surface.addEventListener("pointercancel",stop);
    surface.addEventListener("dblclick",()=>{
      s.zoom=1;s.dx=0;s.dy=0;
      scheduleMPRDraw(plane);
    });
  });

  reloadMPRVolume();
}
"""

print("\n[2/4] Installing new browser MPR engine...")

# This standalone launcher skips the experimental Cell 4D patch.
# Locate the actual 2D function in the generated HTML, not a
# comment marker from a cell that might never have run.
start_candidates = [
    VIEWER_HTML.find("// SINUS SMOOTH MPR ENGINE V2"),
    VIEWER_HTML.find("// SINUS BROWSER-RESIDENT MPR ENGINE"),
    VIEWER_HTML.find("function draw(plane){"),
    VIEWER_HTML.find("function draw(plane) {"),
]
start = min((v for v in start_candidates if v >= 0), default=-1)
end = VIEWER_HTML.find("let vtkFullScreen = null;", max(0, start))

if start < 0 or end <= start:
    raise RuntimeError(
        "Cannot identify 2D/3D JavaScript boundaries; viewer unchanged."
    )
print("2D engine boundary verified:", start, "to", end)

# Preserve JavaScript before and after 2D functions,
# including the working 3D renderer.
VIEWER_HTML = (
    VIEWER_HTML[:start] +
    NEW_MPR_JS + "\n" +
    VIEWER_HTML[end:]
)

# Presets now trigger a full windowed volume refresh,
# not individual PNG requests.
VIEWER_HTML = VIEWER_HTML.replace(
    "planes.forEach(loadSlice);",
    "reloadMPRVolume();"
)

print("[3/4] Verifying engine...")

assert "function initialize3D()" in VIEWER_HTML
assert "function initializePlanes()" in VIEWER_HTML
assert "async function reloadMPRVolume()" in VIEWER_HTML
assert 'src="/vendor/vtk.js"' in VIEWER_HTML

print("2D local-memory engine: INSTALLED")
print("3D vtk.js engine: PRESERVED")

print("\n[4/4] Checking backend...")

r=requests.get(
    "http://127.0.0.1:8765/health",
    timeout=10
)
r.raise_for_status()

print("\n" + "=" * 68)
print("CELL 4E COMPLETE")
print("=" * 68)
print("Backend            : ONLINE")
print("Original resolution:", FULL.shape)
print("MPR volume         :", round(bone_path.stat().st_size/1048576,1),"MiB")
print("Per-slice requests : DISABLED after volume loads")
print("3D viewer         : PRESERVED")
print("External tunnel    : PRESERVED")
print("Original DICOM     : UNCHANGED")
print("Preparation time   :", round(time.time()-start_time,1),"seconds")
print()
print("Refresh browser using Ctrl+Shift+R.")
print("Wait for LOCAL MPR READY before testing scrolling.")
print("=" * 68)



# ===================== FROM VERIFIED SOURCE CELL 9 =====================

# ============================================================
# SINUS VIEWER PRO — CELL 4F
# COMPRESSED BROWSER VOLUME DELIVERY
# KEEP SAME SMOOTH CELL 4E VIEWER
# ============================================================

import os
import time
import zlib
from pathlib import Path
from fastapi import HTTPException
from fastapi.responses import FileResponse
from fastapi.routing import APIRoute

print("=" * 65)
print("SINUS VIEWER — DOWNLOAD OPTIMIZATION")
print("=" * 65)

assert "APP" in globals(), "Cell 4 required"
assert "build_mpr_file" in globals(), "Cell 4E required"
assert SINUS_SERVER.started, "Server not running"

# Confirm existing MPR endpoint.
old_routes = [
    r for r in APP.router.routes
    if isinstance(r, APIRoute)
    and r.path == "/api/mpr-volume"
]

if not old_routes:
    raise RuntimeError("Cell 4E endpoint not found.")

COMPRESS_DIR = Path("/content/sinus_viewer_runtime/mpr")
COMPRESS_DIR.mkdir(parents=True, exist_ok=True)

def prepare_compressed(path):
    source = Path(path)
    target = Path(str(source) + ".gz")

    if (target.exists()
        and target.stat().st_mtime >= source.stat().st_mtime):
        return target

    temp = Path(str(target) + ".pending")

    try:
        compressor = zlib.compressobj(
            level=1,
            wbits=16 + zlib.MAX_WBITS
        )

        with open(source, "rb") as inp, open(temp, "wb") as out:
            while True:
                block = inp.read(4 * 1024 * 1024)
                if not block:
                    break
                out.write(compressor.compress(block))

            out.write(compressor.flush())

        os.replace(temp, target)
        return target
    finally:
        if temp.exists():
            temp.unlink()

print("\n[1/3] Testing compression on Bone preset...")
t0 = time.time()

source = build_mpr_file(400, 2000)
compressed = prepare_compressed(source)

original_mb = source.stat().st_size / 1048576
compressed_mb = compressed.stat().st_size / 1048576

ratio = compressed_mb / original_mb
USE_COMPRESSION = ratio < 0.90

print("Original volume :", round(original_mb, 2), "MiB")
print("Compressed      :", round(compressed_mb, 2), "MiB")
print("Size reduction  :", round((1-ratio)*100, 1), "%")
print("Preparation     :", round(time.time()-t0, 2), "s")

# Restore original route if the compression patch is rerun.
for route in old_routes:
    APP.router.routes.remove(route)

print("\n[2/3] Installing optimized endpoint...")

@APP.get("/api/mpr-volume")
def optimized_mpr_volume(level: int = 400, width: int = 2000):
    if (level, width) not in MPR_PRESETS:
        raise HTTPException(400, "Unsupported CT preset")

    original = build_mpr_file(level, width)

    if USE_COMPRESSION:
        packed = prepare_compressed(original)
        return FileResponse(
            packed,
            media_type="application/octet-stream",
            headers={
                "Content-Encoding": "gzip",
                "Cache-Control": "private, no-store",
                "X-Content-Type-Options": "nosniff"
            }
        )

    return FileResponse(
        original,
        media_type="application/octet-stream",
        headers={
            "Cache-Control": "private, no-store",
            "X-Content-Type-Options": "nosniff"
        }
    )

print("[3/3] Checking endpoint configuration...")

assert any(
    isinstance(r, APIRoute)
    and r.path == "/api/mpr-volume"
    for r in APP.router.routes
)

print("\n" + "=" * 65)
print("CELL 4F COMPLETE")
print("=" * 65)
print("Delivery mode       :", "GZIP" if USE_COMPRESSION else "RAW")
print("Transfer reduction  :", round((1-ratio)*100, 1), "%")
print("3D engine           : UNCHANGED")
print("2D resolution       : UNCHANGED")
print("Existing tunnel     : UNCHANGED")
print("Original DICOM      : UNCHANGED")
print()
print("Refresh browser to test download performance.")
print("=" * 65)



# ===================== FROM VERIFIED SOURCE CELL 10 =====================

# ============================================================
# SINUS 3D VIEWER PRO — CELL 5
# TEMPORARY HTTPS LINK + SESSION SECURITY
# ============================================================

import os
import re
import time
import secrets
import hmac
import subprocess
import urllib.request
from pathlib import Path

import requests
from IPython.display import display, HTML

print("=" * 68)
print("SINUS 3D VIEWER PRO — EXTERNAL BROWSER LAUNCHER")
print("=" * 68)

# 1. Verify Colab backend
if "SINUS_SERVER" not in globals() or not SINUS_SERVER.started:
    raise RuntimeError("Cell 4 backend not running.")

local_url = "http://127.0.0.1:8765"

response = requests.get(
    local_url + "/health",
    timeout=10
)
response.raise_for_status()

print("\n[1/5] Backend verified:", response.json())

# 2. Rotate previous credentials
VIEWER_PASSWORD = secrets.token_urlsafe(24)
VIEWER_SESSION = secrets.token_urlsafe(32)

print("[2/5] New private session credentials generated.")
print("Previous password and login cookies invalidated.")

# 3. Download Cloudflare's official tunnel client
binary = Path("/content/cloudflared")

if not binary.exists():
    print("[3/5] Downloading Cloudflare tunnel client...")

    url = (
        "https://github.com/cloudflare/cloudflared/"
        "releases/latest/download/cloudflared-linux-amd64"
    )

    urllib.request.urlretrieve(url, str(binary))
    binary.chmod(0o755)
else:
    print("[3/5] Existing Cloudflare client found.")

version = subprocess.run(
    [str(binary), "--version"],
    capture_output=True,
    text=True,
    timeout=15,
    check=True
)
print(version.stdout.strip())

# 4. Stop previous tunnel if any
if "SINUS_TUNNEL" in globals():
    if SINUS_TUNNEL.poll() is None:
        SINUS_TUNNEL.terminate()
        try:
            SINUS_TUNNEL.wait(timeout=8)
        except subprocess.TimeoutExpired:
            SINUS_TUNNEL.kill()
            SINUS_TUNNEL.wait(timeout=5)

print("[4/5] Starting temporary HTTPS tunnel...")

log_path = Path("/content/sinus_viewer_tunnel.log")

log_handle = open(log_path, "w", encoding="utf-8")

SINUS_TUNNEL = subprocess.Popen(
    [
        str(binary),
        "tunnel",
        "--no-autoupdate",
        "--protocol", "http2",
        "--url", local_url
    ],
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    start_new_session=True
)

log_handle.close()

# Discover actual URL from tunnel logs
public_url = None
pattern = re.compile(
    r"https://[a-zA-Z0-9-]+\.trycloudflare\.com"
)

for attempt in range(35):
    if SINUS_TUNNEL.poll() is not None:
        break

    contents = log_path.read_text(
        encoding="utf-8",
        errors="replace"
    )

    match = pattern.search(contents)

    if match:
        public_url = match.group(0)
        break

    time.sleep(1)

if not public_url:
    print("\nTunnel log (last lines):")
    print("\n".join(
        log_path.read_text(errors="replace").splitlines()[-15:]
    ))
    raise RuntimeError(
        "External URL not generated. Tunnel may be unavailable "
        "or blocked by this runtime."
    )

SINUS_PUBLIC_URL = public_url

# 5. Test external endpoint without authenticating
print("[5/5] Checking external HTTPS endpoint...")

external_status = "NOT VERIFIED"

for attempt in range(8):
    try:
        r = requests.get(
            public_url + "/login",
            timeout=12,
            allow_redirects=False
        )

        if r.status_code == 200 and "Sinus Viewer Login" in r.text:
            external_status = "VERIFIED"
            break

    except requests.RequestException:
        pass

    time.sleep(2)

print("\n" + "=" * 68)
print("SINUS 3D VIEWER — EXTERNAL ACCESS")
print("=" * 68)
print("Backend         : ONLINE")
print("Temporary link  :", SINUS_PUBLIC_URL)
print("HTTPS status    :", external_status)
print("Patient files   : UNCHANGED")
print()
print("NEW PRIVATE PASSWORD:")
print(VIEWER_PASSWORD)
print()
print("Keep this password private.")
print("Do not share screenshots containing the password.")
print("=" * 68)

if external_status == "VERIFIED":
    display(HTML(
        '<div style="padding:15px;background:#102235;'
        'border:1px solid #387fc0;border-radius:12px">'
        '<h3 style="color:#e9f4ff">Sinus Viewer Ready</h3>'
        f'<a href="{SINUS_PUBLIC_URL}" target="_blank" '
        'rel="noopener noreferrer" '
        'style="color:#72b7ff;font-size:16px">'
        'OPEN SINUS 3D VIEWER ↗</a>'
        '</div>'
    ))
else:
    print(
        "Tunnel URL generated, but external access could not "
        "be verified. Do not assume it is working yet."
    )

print("\nCELL 5 COMPLETE")



SINUS 3D VIEWER PRO — INITIAL SETUP

SYSTEM INFORMATION
-----------------------------------------------------------------
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB
GPU Memory: 79.25 GB
System RAM: 167.05 GB
SimpleITK: 2.5.6

GOOGLE DRIVE
-----------------------------------------------------------------
Mounted: True

ORIGINAL DATASET SAFETY
Read-only inspection policy: ENABLED
No patient files modified.

CELL 1 COMPLETE
SINUS 3D VIEWER PRO — PATIENT DICOM SCANNER

Paste patient folder FULL PATH.
Example: /content/drive/MyDrive/Sinus_Project/Sinus/PATIENT_FOLDER

PATIENT FOLDER PATH: /content/drive/MyDrive/Sinus_Project/New Sinus/101-110/ASHWANI_K_3819047

[1/5] Scanning folders...

[2/5] Available DICOM series
----------------------------------------------------------------------
[0] 641 files | Folder: ASHWANI_K_3819047
----------------------------------------------------------------------
Total series: 1
Folders scanned: 1
Folders with scan